## Predicting Solar, Total Power Load, and Energy Price with LightGBM Model incorporating CAMS data

Here I will analyse an energy dataset provided by Kolasniwash on Kaggle to predict solar generation as well as power demand and price in Spain (see, <https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather/data>). For solar power generation, this will be predicted by incorporating many different weather and time features as well as CAMS solar radiation time-series data with cloud features (see, <https://ads.atmosphere.copernicus.eu/datasets/cams-solar-radiation-timeseries?tab=overview>). Next I will predict total power load and total power generation independently and then predict price using the predicted power load and power generation while also incorporating many different weather and time features. The ML model I will be using for all of the predicitons will be LightGBM, a gradient boosted decision tree model.

In [1]:
# Import the essential Python libraries we will definitely be using for this analysis.
import numpy as np
import pandas as pd
from pathlib import Path
import os
import csv
from datetime import date, datetime, timezone          # To help deal with time series data.
import math
from tqdm import tqdm                  # Progress bar for loops that take a while.

# Import visualization libraries, I'm a big fan of Bokeh, but will also make use of Matplotlib.
# Matplotlib plotting library and functions.
import matplotlib as mp
import matplotlib.pyplot as plt
import seaborn as sns

# Bokeh plotting library and functions. Note, this is probably an overkill but I have used most of these in my other projects.
from bokeh.io import output_notebook, show
from bokeh.models.annotations.labels import Label
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, CustomJS, Slider, Whisker, BoxAnnotation, Arrow, OpenHead, Span
from bokeh.plotting import figure, show, output_file, save
from bokeh.models import Legend, LinearAxis, Range1d, ColumnDataSource, LabelSet, HoverTool, DatetimeTickFormatter

import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from scipy.optimize import curve_fit
from loess.loess_1d import loess_1d
from bokeh.palettes import Category20
import pvlib
from pvlib.location import Location
import pgeocode
import holidays
from suntime import Sun
import lightgbm as lgb
from tqdm.auto import tqdm
import dill

In [2]:
# Get current directory. I will use this to construct the file path to the data files.
current_dir = str(os.getcwd())

files_dir = current_dir + '/data/'

# dataframe holding power usage/generation data. Use default options unless we encounter issues.
power_pd = pd.read_csv(files_dir+'energy_dataset.csv', header=0)

# dataframe holding weather info. Use default options unless we encounter issues.
weather_pd = pd.read_csv(files_dir+'weather_features.csv', header=0)

In [2]:
def load_cams_hourly_csvs(
    files_dir,
    pattern="cams_solar_rad_weather_*_hourly.csv",
    parse_dates=("utc_time",),
    enforce_columns=True,
):
    """
    Read all CAMS hourly CSVs in `files_dir` matching the naming pattern
    'cams_solar_rad_weather_city_name_hourly.csv' into a single DataFrame.

    Parameters
    ----------
    files_dir : str or Path
        Directory containing the CSV files.
    pattern : str
        Glob pattern for matching files.
    parse_dates : tuple[str]
        Columns to parse as datetimes.
    enforce_columns : bool
        If True, enforce the expected column order and presence.

    Returns
    -------
    df_all : pandas.DataFrame
        Concatenated dataframe from all matching files.
    """
    files_dir = Path(files_dir)
    csv_files = sorted(files_dir.glob(pattern))

    if not csv_files:
        raise FileNotFoundError(f"No files found in {files_dir} matching pattern: {pattern}")

    expected_cols = [
        "utc_time",
        "Clear sky GHI",
        "GHI",
        "Solar_capacity",
        "Cloud optical depth",
        "Cloud coverage",
        "Snow probability",
        "Cloud type",
        "daytime",
        "loc_lat",
        "loc_long",
        "city_name",
        "distance",
        "daytime_frac",
        "daytime_any",
    ]

    dfs = []
    for fp in csv_files:
        df = pd.read_csv(fp, encoding="utf-8", parse_dates=list(parse_dates))

        # If city_name column is missing (shouldn't be, per your spec), infer from filename
        if "city_name" not in df.columns:
            city = fp.stem.replace("cams_solar_rad_weather_", "").replace("_hourly", "")
            df["city_name"] = city.replace("_", " ")

        if enforce_columns:
            missing = [c for c in expected_cols if c not in df.columns]
            extra = [c for c in df.columns if c not in expected_cols]
            if missing:
                raise ValueError(f"{fp.name}: missing columns: {missing}")
            if extra:
                # Not fatal; keep extras at the end if you want, but default is to error.
                raise ValueError(f"{fp.name}: unexpected extra columns: {extra}")

            df = df[expected_cols]  # enforce consistent order

        dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)

    # Helpful: sort for time-series work
    if "utc_time" in df_all.columns:
        df_all = df_all.sort_values(["city_name", "utc_time"]).reset_index(drop=True)

    return df_all

In [4]:
# Read in the CAMS Solar radiation and weather data into a single pandas dataframe.
cams_hourly_pd = load_cams_hourly_csvs(files_dir)

In [3]:
# Let's visualize power data by plotting power generation/demand for the various power sources along with
# some optional weather features.

# Set up plotting figure. Set x-axis to "datetime" so that the date time can be displayed appropriately.
def explore_plots(dataframe, x_axis_column, y_axis_columns, x_axis_label, y_axis_label, title, feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False, color_plot='black', color_features='black',
                  labels=None, features_labels=None, symbols=None, symbols_features=None, replaced_columns=None):

    """
    Function to create interactive exploratory plots.
    
    Parameters
    ----------
    dataframe : Pandas dataframe object
        The Pandas dataframe holding the data you want to plot.
    x_axis_column : string
        The name of the column you want to plot along the x-axis.
    y_axis_columns : list
        The names of the columns for the primary data you want to plot along the y-axis.
    x_axis_label : string
        The label to use for the x axis.
    y_axis_label : string
        The label to use for the y axis for the primary data. Note, all column data plotted will have the same y-axis scale
        and label name.
    title : string
        The title for the plot.

    Optional
    --------
    feature_columns : list or string
        The names of the columns you want to plot as additional features along secondary y-axis
        Default: None
    features_ylabel : list or string
        The secondary y-axis feature label/s. If a list greater than one item, list is concatenated into a single string.
        Default: None    
    p : bokeh plotting object
        A bokeh plotting object to add additional plotting objects to.
        Default: None
    normalize : Boolean
        Whether to normalize the data. Caution, might not work well for some features or when including more than one feature.
        Default: False
    other_colors : Boolean
        Whether to use your own colors (True) or to use color names as defined in this function (False).
        Default: False
    color_plot : string or list
        A string or list of colors to use for plotting the primary data. If list, must be length of number y_axis_columns.
        Default: 'black'
    color_features : string or list
        A string or list of colors to use for plotting the features data. If list, must be length of number feature_columns.
        Default: 'black'
    labels : list or string
        The labels to assign all the primary data from the y_axis_columns, should be list of length y_axis_columns if more than one
        primary data is being plotted.
        Default: None
    features_labels : list or string
        The labels to assign all the features data from feature_columns, should be list of length feature_columns if more than one
        feature is being plotted.
        Default: None
    symbols : string or list
        The plotting markers for the primary data. If list, must be length of number of y_axis_columns.
        Default: None
    symbols_features : string or list
        The plotting markers for the features data. If list, must be length of number of feature_columns.
        Default: None
    replaced_columns : Boolean
        Whether to plot the replaced primary data produced through cleaning. The appropriate columns must exist in the dataframe
        if True and given as 'replaced_' and the y_axis_columns, e.g., 'replaced_energy_usage_Wh' for replacements for 
        the 'energy_usage_Wh'.
        Default: False

    Returns
    -------
    p : bokeh object
        The Bokeh plotting object.
    """
    
    colors={'violet':'#6E36BB','pink':'#D8BAFF','blue':'#2480D0','cyan':'#00E6E6','green':'#1DD14B',
            'yellow':'#FFD700','orange':'#FF6600','dorange':'#DAA520','red':'#DD082C','black':'#000000',
            'grey':'#D0D0D0','dgrey':'#666666'}

    if p is None:
        p = figure(title=title, height=900, width=1600, x_axis_type="datetime",
                   tools="reset, hover, zoom_in, zoom_out, box_zoom, wheel_zoom, pan, save")
    
        p.title.text_font_size = '20pt'
        p.yaxis.axis_label = y_axis_label
        p.xaxis.axis_label_text_font_size = "20pt"
        p.xaxis.major_label_text_font_size = "20pt"
        p.xaxis.axis_label_text_font = "times"
        p.xaxis.axis_label_text_color = "black"
        p.xaxis.major_tick_in = 10
        p.xaxis.major_tick_out = 0
        p.xaxis.minor_tick_in = 4
        p.xaxis.minor_tick_out = 0
        p.xaxis.major_tick_line_width = 2
        p.xaxis.axis_label = x_axis_label
        p.yaxis.axis_label_text_font_size = "20pt"
        p.yaxis.major_label_text_font_size = "20pt"
        p.yaxis.axis_label_text_font = "times"
        p.yaxis.axis_label_text_color = "black"
        p.yaxis.major_tick_in = 10
        p.yaxis.major_tick_out = 0
        p.yaxis.minor_tick_in = 4
        p.yaxis.minor_tick_out = 0
        p.yaxis.major_tick_line_width = 2
        # Rotate labels for better readability
        p.xaxis.major_label_orientation = 120
        # Reduce the number of x-axis ticks to avoid crowding
        p.xaxis.ticker.desired_num_ticks = 8

        # Format the x-axis datetime labels.
        p.xaxis.formatter = DatetimeTickFormatter(
            minutes="%d-%m-%y %H:%M",
            hours="%d-%m-%y %H:%M",
            days="%d-%m-%y %H:%M",
            months="%d-%m-%y %H:%M",
            years="%d-%m-%y %H:%M"
        )

    if other_colors:
        color_plot_values = color_plot
    else:
        if isinstance(color_plot, list): 
            color_plot_values = [colors[c] for c in color_plot]
        else:
            color_plot_values = colors[color_plot]

    # Create a loop to plot each column data as given by y_axis_columns. Note that these columns will have the same y-range along
    # the primary y-axis.
    color_index = 0
    marker_index = 0
    label_index = 0

    # print('labels: ', labels)
    # print('markers: ', symbols)
    # print('colors: ', color_plot_values)
    
    for i, column in enumerate(y_axis_columns):
        label = labels[label_index]

        # Create a loop to plot the column data for each city in Spain
        for y, city in enumerate(dataframe['city_name'].unique()):

            if normalize:
                max_primary_data = dataframe.loc[dataframe['city_name'] == city, column].max()
            else:
                max_primary_data = 1

            # Only show the first column by default, hide others
            visible = True if i == 0 | y == 0 else False
            
            # Scatter plot with symbols.
            if symbols:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data, size=10,
                          marker=symbols[marker_index], color=color_plot_values[color_index], alpha=0.5,
                          legend_label=label + ' ' + str(city), visible=visible)
                
            # Add a line to connect the symbols
            p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                   dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data,
                   line_width=2, color=color_plot_values[color_index], alpha=0.5, legend_label=label + ' ' + str(city),
                   visible=visible)

            if replaced_columns:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, f'replaced_{column}']/max_primary_data,
                          size=10, color="red", marker="diamond", alpha=0.5, legend_label="Replaced " + label + ' ' + str(city),
                          visible=visible)

            # print('marker_index: ', marker_index)
            # print('color_index: ', color_index)
            # print('label_index: ', label_index)
        
            marker_index = marker_index + 1
            color_index = color_index + 1
        label_index = label_index + 1
        

    # Create a loop to plot the feature data, if given. Note, secondary y axis must be the same for all features!
    if feature_columns:

        if other_colors:
            color_plot_values = color_features
        else:
            if isinstance(color_features, list): 
                color_plot_values = [colors[c] for c in color_features]
            else:
                color_plot_values = colors[color_features]

        if normalize:
            max_features_data = dataframe[feature_columns].max().max()
        else:
            max_features_data = 1

        # Specify the secondary y-range for plotting the features data. All features data will be plotted on the same secondary
        # y-axis range, so find minimum and maximum values for the first feature
        y_range = p.y_range
        p.extra_y_ranges['features'] = y_range
        label_index = 0
        color_index = 0
        marker_index = 0

        for i, feature_column in enumerate(feature_columns):

            feature_label = features_labels[label_index]

            # Create a loop to plot the features for each city.
            for y, city in enumerate(dataframe['city_name'].unique()):

                if normalize:
                    max_features_data = dataframe.loc[dataframe['city_name'] == city, feature_column].max()
                else:
                    max_features_data = 1

                # Only show the first column by default, hide others
                visible = True if i == 0 | y == 0 else False
                
                # Scatter plot with symbols. Only plot the energy usage as a function of time for a specific city.
                if symbols_features:
                    p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                              dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data,
                              y_range_name="features", size=10, marker=symbols_features[marker_index],
                              color=color_plot_values[color_index], alpha=0.5, legend_label=feature_label + ' ' + str(city),
                              visible=visible)
                # Add a line to connect the symbols
                p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                       dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data, y_range_name="features",
                       line_width=2, color=color_plot_values[color_index], alpha=0.5,
                       legend_label=feature_label + ' ' + str(city), visible=visible)
            
                marker_index = marker_index + 1
                color_index = color_index + 1
            label_index = label_index + 1

        # Add the secondary y-axis
        secondary_y_axis = LinearAxis(y_range_name="features", axis_label=" ".join(features_ylabel))
        p.add_layout(secondary_y_axis, 'right')

        secondary_y_axis.axis_label_text_font_size = "20pt"  # Adjust label font size
        secondary_y_axis.major_label_text_font_size = "20pt"  # Adjust tick label font size
        secondary_y_axis.axis_label_text_font = "times"
        secondary_y_axis.axis_label_text_color = "black"
        secondary_y_axis.axis_label_text_font_size = "16pt"
        secondary_y_axis.axis_label_text_font_style = "normal"
        secondary_y_axis.major_tick_in = 10
        secondary_y_axis.major_tick_out = 0
        secondary_y_axis.minor_tick_in = 4
        secondary_y_axis.minor_tick_out = 0

    # Allow user to hide/show plot features.
    p.add_layout(Legend(), 'right')
    p.legend.click_policy="hide"
    p.legend.background_fill_alpha = 0.3
    p.legend.border_line_alpha = 0.2

    return(p)

In [6]:
# Let's make copies of power_pd and weather_pd dataframes.
power_pd1 = power_pd.copy(deep=True)
weather_pd1 = weather_pd.copy(deep=True)

# Rename time columns for consistency in the power and weather dataframes.
power_pd1.rename(columns={'time':'local_time'}, inplace=True)
weather_pd1.rename(columns={'dt_iso':'local_time'}, inplace=True)

# Now convert the date/time strings in the local_time and utc_time columns to datetime64 data type so that I can work with
# the time series data.
power_pd1['local_time_tz'] = pd.to_datetime(power_pd1['local_time'], utc=True)
power_pd1['utc_time'] = power_pd1['local_time_tz'].dt.tz_localize(None)
power_pd1['local_time'] = power_pd1['local_time_tz'].dt.tz_convert('Europe/Madrid').dt.tz_localize(None)
power_pd1.drop(columns='local_time_tz', inplace=True)
column_utc_time = power_pd1.pop('utc_time')
power_pd1.insert(1, 'utc_time', column_utc_time)

weather_pd1['local_time_tz'] = pd.to_datetime(weather_pd1['local_time'], utc=True)
weather_pd1['utc_time'] = weather_pd1['local_time_tz'].dt.tz_localize(None)
weather_pd1['local_time'] = weather_pd1['local_time_tz'].dt.tz_convert('Europe/Madrid').dt.tz_localize(None)
weather_pd1.drop(columns='local_time_tz', inplace=True)
column_utc_time = weather_pd1.pop('utc_time')
weather_pd1.insert(1, 'utc_time', column_utc_time)

In [7]:
# A column for city names in the power dataset and set to 'Spain' for now.
power_pd1['city_name'] = 'Spain'
column_city_name = power_pd1.pop('city_name')
power_pd1.insert(2, 'city_name', column_city_name)

In [8]:
power_pd2 = power_pd1.copy(deep=True)

# Identify generation columns where all values are NaN and/or 0
null_zero = []
for col in power_pd2.columns:
    if col.startswith('generation'):
        # Create boolean mask where values are NaN or 0
        is_null_or_zero = power_pd2[col].isna() | (power_pd2[col] == 0.0)
        if is_null_or_zero.all():
            null_zero.append(col)

# Print and drop those columns
print("Columns with all null/zero values:")
for col in null_zero:
    print(col)

# Drop from power_pd2
power_pd2.drop(columns=null_zero, inplace=True)
power_pd2.drop(columns=['forecast wind offshore eday ahead'], inplace=True)

Columns with all null/zero values:
generation fossil coal-derived gas
generation fossil oil shale
generation fossil peat
generation geothermal
generation hydro pumped storage aggregated
generation marine
generation wind offshore


In [4]:
# Weather severity mapping that is used to help select the most severe weather conditions from two duplicate
# time stamps.

def get_weather_severity(weather_id):
    if 200 <= weather_id < 300:
        return 5  # Thunderstorm
    elif 300 <= weather_id < 400:
        return 3  # Drizzle
    elif 500 <= weather_id < 600:
        return 4  # Rain
    elif 600 <= weather_id < 700:
        return 4  # Snow
    elif 700 <= weather_id < 800:
        return 2  # Atmosphere (mist, fog)
    elif weather_id == 800:
        return 0  # Clear
    elif 801 <= weather_id <= 804:
        return 1  # Clouds
    else:
        return -1  # Unknown or invalid

In [10]:
weather_pd2 = weather_pd1.copy(deep=True)

weather_pd2['severity'] = weather_pd2['weather_id'].apply(get_weather_severity)

# Sort by severity and weather_id
weather_pd2 = weather_pd2.sort_values(by=['city_name', 'utc_time', 'severity', 'weather_id'], ascending=[True, True, False, False])

# Identify the index of the rows that will be kept
keep_index = weather_pd2.drop_duplicates(subset=['city_name', 'utc_time'], keep='first').index

# Create the audit dataframe, mark rows as kept or dropped
weather_duplicates_audit = weather_pd2.copy(deep=True)
weather_duplicates_audit['keep_row'] = weather_duplicates_audit.index.isin(keep_index)

# Drop duplicates keeping most severe (and then highest ID) row
weather_pd2 = weather_pd2.drop_duplicates(subset=['city_name', 'utc_time'], keep='first').reset_index(drop=True)
weather_duplicates_audit.reset_index(drop=True, inplace=True)

for city in weather_pd2['city_name'].unique():
    print('Number of utc time rows in the weather dataset: ', len(weather_pd2.loc[weather_pd2['city_name'] == city, 'utc_time']),
         ' in city: ', city)
    print('Number of unique utc time rows in the weather dataset: ',
          len(weather_pd2.loc[weather_pd2['city_name'] == city, 'utc_time'].unique()),
         ' in city: ', city)
    print(' ')

Number of utc time rows in the weather dataset:  35064  in city:   Barcelona
Number of unique utc time rows in the weather dataset:  35064  in city:   Barcelona
 
Number of utc time rows in the weather dataset:  35064  in city:  Bilbao
Number of unique utc time rows in the weather dataset:  35064  in city:  Bilbao
 
Number of utc time rows in the weather dataset:  35064  in city:  Madrid
Number of unique utc time rows in the weather dataset:  35064  in city:  Madrid
 
Number of utc time rows in the weather dataset:  35064  in city:  Seville
Number of unique utc time rows in the weather dataset:  35064  in city:  Seville
 
Number of utc time rows in the weather dataset:  35064  in city:  Valencia
Number of unique utc time rows in the weather dataset:  35064  in city:  Valencia
 


In [5]:
from scipy.optimize import curve_fit

# Sine-like daily solar generation curve
def daily_solar_curve(hour, amplitude, phase_shift, vertical_shift):
    return amplitude * np.sin((np.pi / 12) * (hour - phase_shift)) + vertical_shift


def get_median_night_residual(df, date_col, hour_col, solar_col, target_date):
    """
    Returns the median nighttime value for the closest night to target_date in the dataframe.
    Nighttime is defined as hours < 6 or >= 18.
    """
    df_night = df[
        ((df[hour_col] < 6) | (df[hour_col] >= 18)) &
        (df[date_col].isin([target_date - pd.Timedelta(days=1), target_date, target_date + pd.Timedelta(days=1)]))
    ]
    # Select night closest to target_date
    night_dates = df_night[date_col].unique()
    if len(night_dates) == 0:
        return 0  # Fallback
    closest_night = min(night_dates, key=lambda d: abs((d - target_date).days))
    return df_night.loc[df_night[date_col] == closest_night, solar_col].median()

def fill_solar_with_rolling_fit(df, time_col, solar_col, filled_col='filled_generation solar',
                                replaced_col='replaced_generation_solar'):
    """
    Fills missing values in the solar generation column using a rolling 3-day window sinusoidal fit.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataframe containing the power generation data.
    time_col : str
        Name of the datetime column (must be datetime64).
    solar_col : str
        Name of the solar generation column.
    filled_col : str
        Name of the column where the filled (original + replacement) data is stored.
    replaced_col : str
        Name of the column where only the replacement values are stored.

    Returns
    ----------
    filled_data : pd.Series
        The column with the filled (original + replacement) data.
    replaced_data : pd.Series
        A column with only the replaced values (NaN elsewhere).
    """
    df = df.copy()
    df['hour'] = df[time_col].dt.hour
    df['date'] = df[time_col].dt.date
    filled_data = df[filled_col].copy()           # Column with the filled data
    replaced_data = df[replaced_col].copy()       # Column with the replaced data

    unique_dates = np.array(sorted(df['date'].unique()))
    is_daytime = lambda h: (h >= 6) & (h <= 18)

    # Iterate through center days for a 3-day rolling window
    for i in range(1, len(unique_dates) - 1):
        window_dates = unique_dates[i-1:i+2]
        center_day = unique_dates[i]

        mask_window = df['date'].isin(window_dates)
        window_df = df[mask_window & df[filled_col].notna()]
        window_df = window_df[is_daytime(window_df['hour'])]
        #window_df = window_df[(window_df['hour'] >= 6) & (window_df['hour'] <= 20)]

        if len(window_df) < 18:
            continue

        try:
            popt, _ = curve_fit(
                daily_solar_curve,
                window_df['hour'],
                window_df[filled_col],
                p0=[window_df[filled_col].max(), 12, 0],
                maxfev=10000
            )
        except Exception:
            continue

        mask_center = (
            (df['date'] == center_day) &
            (df[filled_col].isna()) &
            is_daytime(df['hour'])
        )

        # Identify consecutive NaN gaps (within the center day daytime)
        subidx = df[mask_center].index
        # We'll group gaps by consecutive indices (daytime NaNs)
        if not subidx.empty:
            groups = np.split(subidx, np.where(np.diff(subidx) != 1)[0]+1)
            for group in groups:
                if len(group) <= 2:
                    # Interpolate for short gaps
                    interp_vals = filled_data.interpolate(method='linear').loc[group]
                    filled_data.loc[group] = interp_vals
                    replaced_data.loc[group] = interp_vals
                else:
                    # Use curve fit for longer gaps
                    hours = df.loc[group, 'hour']
                    fitted_vals = daily_solar_curve(hours, *popt)
                    max_val = window_df[filled_col].max()
                    # Use median night value for negative fits
                    for idx, val, hr in zip(group, fitted_vals, hours):
                        if val < 0:
                            median_night_val = get_median_night_residual(df, 'date', 'hour', filled_col, center_day)
                            filled_data.loc[idx] = median_night_val
                            replaced_data.loc[idx] = median_night_val
                        else:
                            clipped_val = min(val, max_val)
                            filled_data.loc[idx] = clipped_val
                            replaced_data.loc[idx] = clipped_val

    return filled_data, replaced_data

In [12]:
# Using the hybrid approach with local regression (LOESS) to fill in large gaps.
# I will use loess_1d function from loess as part of the Pypi library (https://pypi.org/project/loess/).
from loess.loess_1d import loess_1d

# Rolling average window size.
rolling_window = 5

# Make a deep copy of the dataframe in case of accidental modifications
power_pd_clean = power_pd2.copy(deep=True)

columns_to_clean = [
    'generation biomass',
    'generation fossil brown coal/lignite',
    'generation fossil gas',
    'generation fossil hard coal',
    'generation fossil oil',
    'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage',
    'generation hydro water reservoir',
    'generation nuclear',
    'generation other',
    'generation other renewable',
    'generation solar',
    'generation waste',
    'generation wind onshore',
    'forecast solar day ahead',
    'forecast wind onshore day ahead',
    'total load forecast',
    'total load actual',
    'price day ahead',
    'price actual'
]

# Ensure datetime column is available as integer timestamps for LOESS
power_pd_clean['utc_timestamp'] = power_pd_clean['utc_time'].astype('int64') // 10**9

# Replace negative values with NaN (assumed invalid)
for col in columns_to_clean:
    power_pd_clean.loc[power_pd_clean[col] < 0, col] = np.nan

# Apply hybrid imputation to each column
for col in columns_to_clean:
    print(f"Processing column: {col}")
    
    # Create new columns to track replacement and filled values
    power_pd_clean[f'replaced_{col}'] = np.where(power_pd_clean[col].isna(), True, np.nan)
    power_pd_clean[f'filled_{col}'] = power_pd_clean[col].copy()

    #Need to handle solar power generation separately to the other columns as the LOESS approach does not work well.
    if col == 'generation solar':
        # Use the fill_solar_with_rolling_fit function to fill the missing values for the solar generation column.
        solar_filled, solar_replaced = fill_solar_with_rolling_fit(power_pd_clean, 'local_time', 'generation solar',
                                                                   filled_col='filled_generation solar',
                                                                   replaced_col='replaced_generation solar')
        power_pd_clean[f'filled_{col}'] = solar_filled
        power_pd_clean[f'replaced_{col}'] = solar_replaced
        
    else:
        # Step 1: Rolling average for small gaps
        rolling_filled = power_pd_clean[col].rolling(window=rolling_window, center=True, min_periods=1).mean()
        small_gap_mask = power_pd_clean[col].isna()
        power_pd_clean.loc[small_gap_mask, f'filled_{col}'] = rolling_filled[small_gap_mask]
        power_pd_clean.loc[small_gap_mask, f'replaced_{col}'] = rolling_filled[small_gap_mask]
      
        # Step 2: LOESS for medium gaps
        subset = power_pd_clean.copy()
        subset['utc_timestamp'] = power_pd_clean['utc_timestamp']
        mask_loess = subset[f'filled_{col}'].isna()
    
        valid_idx = subset[col].notna()
        missing_idx = subset[col].isna()
        loess_frac = 0.0005
    
        if valid_idx.sum() > 10 and missing_idx.sum() > 0:
            valid_timestamps = subset.loc[valid_idx, 'utc_timestamp'].to_numpy()
            valid_values = subset.loc[valid_idx, col].to_numpy()
            missing_timestamps = subset.loc[missing_idx, 'utc_timestamp'].to_numpy()
    
            #print(missing_timestamps)
    
            try:
                time_out, loess_fitted, _ = loess_1d(valid_timestamps, valid_values, xnew=missing_timestamps, frac=loess_frac)
    
                if len(loess_fitted) > 0:
                    loess_df = pd.DataFrame({'utc_timestamp': missing_timestamps, 'loess_value': loess_fitted})
                    loess_df.set_index(subset.loc[missing_idx].index, inplace=True)
        
                    power_pd_clean.loc[mask_loess, f'filled_{col}'] = loess_df['loess_value']
                    power_pd_clean.loc[mask_loess, f'replaced_{col}'] = loess_df['loess_value']
    
            except Exception as e:
                print(f"⚠️ LOESS failed for column '{col}': {e}")
                # Fallback to linear interpolation
                interpolated_values = power_pd_clean[col].interpolate(method='linear')
                power_pd_clean.loc[mask_loess, f'filled_{col}'] = interpolated_values[mask_loess]
                power_pd_clean.loc[mask_loess, f'replaced_{col}'] = interpolated_values[mask_loess]

    # Step 3: Final fallback — linear interpolation
    final_mask = power_pd_clean[f'filled_{col}'].isna()
    power_pd_clean.loc[final_mask, f'filled_{col}'] = power_pd_clean[col].interpolate(method='linear')[final_mask]
    power_pd_clean.loc[final_mask, f'replaced_{col}'] = power_pd_clean[f'filled_{col}'][final_mask]

direct_out = current_dir + '/output'
filename_out = direct_out + '/power_dataset_missing_LOESS_replaced2.csv'
power_pd_clean.to_csv(filename_out, index=False)

Processing column: generation biomass
Processing column: generation fossil brown coal/lignite
⚠️ LOESS failed for column 'generation fossil brown coal/lignite': SVD did not converge in Linear Least Squares
Processing column: generation fossil gas
Processing column: generation fossil hard coal
Processing column: generation fossil oil
Processing column: generation hydro pumped storage consumption


/Users/u8010412/miniconda3/envs/data_science/lib/python3.11/site-packages/loess/loess_1d.py:271: RuntimeWarning: divide by zero encountered in divide
  uu = (aerr/(6*mad))**2      # For a Gaussian: sigma=1.4826*MAD
/Users/u8010412/miniconda3/envs/data_science/lib/python3.11/site-packages/loess/loess_1d.py:271: RuntimeWarning: invalid value encountered in divide
  uu = (aerr/(6*mad))**2      # For a Gaussian: sigma=1.4826*MAD


Processing column: generation hydro run-of-river and poundage
Processing column: generation hydro water reservoir
Processing column: generation nuclear
Processing column: generation other
Processing column: generation other renewable
Processing column: generation solar
Processing column: generation waste
Processing column: generation wind onshore
Processing column: forecast solar day ahead
Processing column: forecast wind onshore day ahead
Processing column: total load forecast
Processing column: total load actual
Processing column: price day ahead
Processing column: price actual


In [13]:
# Now create a plot indicating the replacements.
from bokeh.palettes import Category20

colors = Category20[20][:20]

# Select 20 distinct Bokeh marker types
available_markers = [
    'circle', 'square', 'triangle', 'diamond', 'inverted_triangle',
    'cross', 'x', 'asterisk', 'circle_cross', 'square_cross',
    'diamond_cross', 'circle_x', 'square_x', 'triangle_dot',
    'plus', 'hex', 'square_dot', 'diamond_dot', 'star', 'star_dot'
]

gen_cols = [
    'generation biomass',
    'generation fossil brown coal/lignite',
    'generation fossil gas',
    'generation fossil hard coal',
    'generation fossil oil',
    'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage',
    'generation hydro water reservoir',
    'generation nuclear',
    'generation other',
    'generation other renewable',
    'generation solar',
    'generation waste',
    'generation wind onshore',
]

forecast_gen_cols = [
    'forecast solar day ahead',
    'forecast wind onshore day ahead'
]

load_cols = [
    'total load actual',
    'total load forecast'
]

price_cols = [
    'price actual',
    'price day ahead'
]

In [14]:
columns_final = ['local_time', 'utc_time', 'utc_timestamp', 'city_name',  
       'filled_generation biomass',
       'filled_generation fossil brown coal/lignite',
       'filled_generation fossil gas',
       'filled_generation fossil hard coal',
       'filled_generation fossil oil',
       'filled_generation hydro pumped storage consumption',
       'filled_generation hydro run-of-river and poundage',
       'filled_generation hydro water reservoir',
       'filled_generation nuclear',
       'filled_generation other',
       'filled_generation other renewable',
       'filled_generation solar',
       'filled_generation waste',
       'filled_generation wind onshore',
       'filled_forecast solar day ahead',
       'filled_forecast wind onshore day ahead',
       'filled_total load forecast',
       'filled_total load actual',
       'filled_price day ahead',
       'filled_price actual']

power_pd_final = power_pd_clean[columns_final].copy(deep=True)

filled_cols = [
    'filled_generation biomass',
    'filled_generation fossil brown coal/lignite',
    'filled_generation fossil gas',
    'filled_generation fossil hard coal',
    'filled_generation fossil oil',
    'filled_generation hydro pumped storage consumption',
    'filled_generation hydro run-of-river and poundage',
    'filled_generation hydro water reservoir',
    'filled_generation nuclear',
    'filled_generation other',
    'filled_generation other renewable',
    'filled_generation solar',
    'filled_generation waste',
    'filled_generation wind onshore',
    'filled_forecast solar day ahead',
    'filled_forecast wind onshore day ahead',
    'filled_total load forecast',
    'filled_total load actual',
    'filled_price day ahead',
    'filled_price actual'
]

rename_dict = {col: col.replace('filled_', '') for col in filled_cols}

power_pd_final.rename(columns=rename_dict, inplace=True)

In [15]:
print(weather_pd2.info())

# Let's do a bit more cleaning of the weather data to ensure there are no nan or bad values.
print(weather_pd2.describe())

nan_counts = weather_pd2.isnull().sum()
print(nan_counts)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175320 entries, 0 to 175319
Data columns (total 19 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   local_time           175320 non-null  datetime64[ns]
 1   utc_time             175320 non-null  datetime64[ns]
 2   city_name            175320 non-null  object        
 3   temp                 175320 non-null  float64       
 4   temp_min             175320 non-null  float64       
 5   temp_max             175320 non-null  float64       
 6   pressure             175320 non-null  int64         
 7   humidity             175320 non-null  int64         
 8   wind_speed           175320 non-null  int64         
 9   wind_deg             175320 non-null  int64         
 10  rain_1h              175320 non-null  float64       
 11  rain_3h              175320 non-null  float64       
 12  snow_3h              175320 non-null  float64       
 13  clouds_all    

In [6]:
# The pressure column has some unrealistic values (0mb for min and 100,837mb for max). The lowest recorded pressure on Earth is
# 870mb in Typhoon Tip and highest recorded pressure is 1084mb. I think a realistic range should be between 930mb to 1060mb.
# The highest recorded wind speed is 113m/s so a realistic maximum should be 50m/s (180km/h).
# rain_3h has many more values that are 0 compared with rain_1h. Therefore, rain_3h will be recomputed by summing rain_1h values
# (two hours before plus current timestamp).

def clean_weather_pd_hourly(
    weather_pd2: pd.DataFrame,
    time_col: str = "utc_time",
    city_col: str = "city_name",
    window_points: int = 25,   # 25 hours ~ ±12 hours centered (odd number required for symmetric center)
    min_periods: int = 7,
    round_pressure_to_int: bool = False,   # set True if you want integer mb after cleaning
    recompute_rain_3h: bool = True,
) -> pd.DataFrame:
    df = weather_pd2.copy()
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.sort_values([city_col, time_col]).reset_index(drop=True)

    if window_points % 2 == 0:
        raise ValueError("window_points must be odd for a centered window (e.g., 25, 49).")

    def _clean_city(g: pd.DataFrame) -> pd.DataFrame:
        city_name = g.name
        g = g.sort_values(time_col).set_index(time_col)

        # Cast to float to avoid dtype assignment warnings
        g["pressure"] = g["pressure"].astype(float)
        g["wind_speed"] = g["wind_speed"].astype(float)
        g["rain_1h"] = g["rain_1h"].astype(float)
        g["rain_3h"] = g["rain_3h"].astype(float)

        # --- pressure: replace <930 or >1060 with rolling median of valid values ---
        p = g["pressure"]
        bad_p = (p < 930.0) | (p > 1060.0)
        p_valid = p.mask(bad_p)  # bad -> NaN so they don't influence the median
        p_med = p_valid.rolling(window_points, center=True, min_periods=min_periods).median()
        p_fill = p_med.interpolate(method="time", limit_direction="both")
        g.loc[bad_p, "pressure"] = p_fill.loc[bad_p]

        # --- wind_speed: replace >50 with rolling median of valid values ---
        w = g["wind_speed"]
        bad_w = w > 50
        w_valid = w.mask(bad_w)
        w_med = w_valid.rolling(window_points, center=True, min_periods=min_periods).median()
        w_fill = w_med.interpolate(method="time", limit_direction="both")
        g.loc[bad_w, "wind_speed"] = w_fill.loc[bad_w]

        # --- recompute rain_3h from rain_1h (current + previous 2 timestamps) ---
        if recompute_rain_3h:
            g["rain_3h"] = g["rain_1h"].rolling(window=3, min_periods=1).sum()

        if round_pressure_to_int:
            g["pressure"] = g["pressure"].round().astype("Int64")  # nullable int
 
        g = g.reset_index()
        g[city_col] = city_name  # add city_name back

        return g

    # Use include_groups=False when available (pandas >= 2.1-ish).
    # Fallback gracefully if older pandas.
    try:
        weather_pd_cleaned = (
            df.groupby(city_col, group_keys=False, sort=False)
              .apply(_clean_city, include_groups=False)
              .reset_index(drop=True)
        )
    except TypeError:
        # older pandas: no include_groups kwarg
        weather_pd_cleaned = (
            df.groupby(city_col, group_keys=False, sort=False)
              .apply(_clean_city)
              .reset_index(drop=True)
        )

    return weather_pd_cleaned

In [ ]:
weather_pd_cleaned = clean_weather_pd_hourly(weather_pd2, window_points=25, min_periods=7, recompute_rain_3h=True)

In [17]:
power_pd_final.drop(columns=['local_time', 'utc_timestamp'], axis=1, inplace=True)
weather_pd_cleaned.drop(columns=['local_time'], axis=1, inplace=True)

In [18]:
weather_pd_cleaned.describe()

,utc_time,temp,temp_min,temp_max,pressure,humidity,wind_speed,wind_deg,rain_1h,rain_3h,snow_3h,clouds_all,weather_id,severity
count,175320,175320.000000,175320.000000,175320.000000,175320.000000,175320.000000,175320.000000,175320.000000,175320.000000,175320.000000,175320.000000,175320.000000,175320.000000,175320.000000
mean,2016-12-31 10:29:59.999999744,289.707651,288.428433,291.172255,1016.051705,68.032307,2.468834,166.724909,0.069308,0.207923,0.004846,24.344057,762.967214,0.887212
min,2014-12-31 23:00:00,262.240000,262.240000,262.240000,930.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000
25%,2016-01-01 04:45:00,283.830000,282.784586,284.909258,1013.000000,53.000000,1.000000,56.000000,0.000000,0.000000,0.000000,0.000000,800.000000,0.000000
50%,2016-12-31 10:30:00,289.150000,288.150000,290.150000,1018.000000,72.000000,2.000000,178.000000,0.000000,0.000000,0.000000,16.000000,800.000000,1.000000
75%,2017-12-31 16:15:00,295.240000,294.150000,297.150000,1022.000000,87.000000,4.000000,270.000000,0.000000,0.000000,0.000000,40.000000,801.000000,1.000000
max,2018-12-31 22:00:00,315.600000,315.150000,321.150000,1048.000000,100.000000,43.000000,360.000000,12.000000,36.000000,21.500000,100.000000,804.000000,5.000000
std,NaN,8.024910,7.948249,8.613916,12.173574,21.838097,2.063790,116.548788,0.385915,0.977268,0.224547,30.339522,104.770349,1.218463


In [19]:
cams_hourly_pd.describe()

,utc_time,Clear sky GHI,GHI,Solar_capacity,Cloud optical depth,Cloud coverage,Snow probability,Cloud type,daytime,loc_lat,loc_long,distance,daytime_frac
count,456144,456144.000000,456144.00000,456144.000000,456144.000000,456144.000000,456144.000000,456144.000000,456144.000000,456144.000000,456144.000000,456144.000000,456144.000000
mean,2016-12-31 00:29:59.999998720,244.095464,206.74611,88.461538,3.151350,15.009102,0.718219,1.164448,0.506123,39.052753,-3.735042,17.202655,0.506123
min,2014-12-31 01:00:00,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,36.660640,-6.744740,1.068655,0.000000
25%,2015-12-31 12:45:00,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,37.579180,-5.390560,2.070380,0.000000
50%,2016-12-31 00:30:00,13.093400,9.53855,100.000000,0.000000,0.000000,0.000000,0.000000,0.583333,39.184090,-3.671640,7.731657,0.583333
75%,2017-12-31 12:15:00,485.182725,377.65430,150.000000,0.062000,3.500000,0.000000,0.000000,1.000000,39.463250,-3.068540,15.218204,1.000000
max,2019-01-01 00:00:00,1073.344200,1073.34400,200.000000,150.872500,100.000000,95.000000,8.000000,1.000000,43.261560,2.168070,111.490328,1.000000
std,NaN,319.158405,291.10725,65.497712,11.795998,31.256351,5.376009,2.513684,0.485672,1.765759,2.365757,28.462905,0.485672


In [21]:
cams_hourly_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 456144 entries, 0 to 456143
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   utc_time             456144 non-null  datetime64[ns]
 1   Clear sky GHI        456144 non-null  float64       
 2   GHI                  456144 non-null  float64       
 3   Solar_capacity       456144 non-null  int64         
 4   Cloud optical depth  456144 non-null  float64       
 5   Cloud coverage       456144 non-null  float64       
 6   Snow probability     456144 non-null  float64       
 7   Cloud type           456144 non-null  int64         
 8   daytime              456144 non-null  float64       
 9   loc_lat              456144 non-null  float64       
 10  loc_long             456144 non-null  float64       
 11  city_name            456144 non-null  object        
 12  distance             456144 non-null  float64       
 13  daytime_frac  

In [19]:
print(min(power_pd_final['utc_time']), max(power_pd_final['utc_time']))

2014-12-31 23:00:00 2018-12-31 22:00:00


In [20]:
# Merge the power data and CAMS into a single dataframe
# First remove rows with times before the first power entry and after the last power entry
min_time = min(power_pd_final['utc_time'])
max_time = max(power_pd_final['utc_time'])
cams_hourly_pd_cut = cams_hourly_pd[~((cams_hourly_pd['utc_time'] < min_time) | (cams_hourly_pd['utc_time'] > max_time))].copy()
cams_hourly_pd_cut.describe()

,utc_time,Clear sky GHI,GHI,Solar_capacity,Cloud optical depth,Cloud coverage,Snow probability,Cloud type,daytime,loc_lat,loc_long,distance,daytime_frac
count,455832,455832.000000,455832.000000,455832.000000,455832.000000,455832.000000,455832.000000,455832.000000,455832.000000,455832.000000,455832.000000,455832.000000,455832.000000
mean,2016-12-31 10:29:59.999998720,244.177329,206.803415,88.461538,3.153487,15.018780,0.718710,1.165109,0.506201,39.052753,-3.735042,17.202655,0.506201
min,2014-12-31 23:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,36.660640,-6.744740,1.068655,0.000000
25%,2016-01-01 04:45:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,37.579180,-5.390560,2.070380,0.000000
50%,2016-12-31 10:30:00,13.135850,9.581250,100.000000,0.000000,0.000000,0.000000,0.000000,0.583333,39.184090,-3.671640,7.731657,0.583333
75%,2017-12-31 16:15:00,485.333200,377.757775,150.000000,0.062500,3.500000,0.000000,0.000000,1.000000,39.463250,-3.068540,15.218204,1.000000
max,2018-12-31 22:00:00,1073.344200,1073.344000,200.000000,150.872500,100.000000,95.000000,8.000000,1.000000,43.261560,2.168070,111.490328,1.000000
std,NaN,319.214681,291.158147,65.497712,11.799751,31.264741,5.377816,2.514282,0.485671,1.765759,2.365757,28.462905,0.485671


In [21]:
merged_power_CAMS = pd.merge(cams_hourly_pd_cut, power_pd_final, on='utc_time', how='left')
merged_power_CAMS.drop(columns=['city_name_y'], axis=1, inplace=True)
merged_power_CAMS.rename(columns={'city_name_x': 'city_name'}, inplace=True)
with pd.option_context('display.max_columns', None):
    print(merged_power_CAMS.columns)

Index(['utc_time', 'Clear sky GHI', 'GHI', 'Solar_capacity',
       'Cloud optical depth', 'Cloud coverage', 'Snow probability',
       'Cloud type', 'daytime', 'loc_lat', 'loc_long', 'city_name', 'distance',
       'daytime_frac', 'daytime_any', 'generation biomass',
       'generation fossil brown coal/lignite', 'generation fossil gas',
       'generation fossil hard coal', 'generation fossil oil',
       'generation hydro pumped storage consumption',
       'generation hydro run-of-river and poundage',
       'generation hydro water reservoir', 'generation nuclear',
       'generation other', 'generation other renewable', 'generation solar',
       'generation waste', 'generation wind onshore',
       'forecast solar day ahead', 'forecast wind onshore day ahead',
       'total load forecast', 'total load actual', 'price day ahead',
       'price actual'],
      dtype='object')


In [22]:
print(merged_power_CAMS['city_name'].unique())
print(weather_pd_cleaned['city_name'].unique())

['Barcelona' 'Bilbao' 'Ecija' 'El Carpio' 'Guadix' 'Logrosan' 'Madrid'
 'San Jose del Valle' 'Santa Marta' 'Seville' 'Tomelloso' 'Valencia'
 'Villarta de San Juan']
[' Barcelona' 'Bilbao' 'Madrid' 'Seville' 'Valencia']


In [23]:
filt = merged_power_CAMS['Solar_capacity'] == 0
filt_list = merged_power_CAMS[filt].copy()
print(filt_list['city_name'].unique())

['Barcelona' 'Bilbao' 'Madrid' 'Valencia']


In [24]:
# Let's drop the cities where energy generation is zero as they likely won't contribute much to the predictive
# modelling.
filt = merged_power_CAMS['Solar_capacity'] != 0
merged_power_CAMS_dropped = merged_power_CAMS[filt].copy()
print(merged_power_CAMS_dropped['city_name'].unique())

['Ecija' 'El Carpio' 'Guadix' 'Logrosan' 'San Jose del Valle'
 'Santa Marta' 'Seville' 'Tomelloso' 'Villarta de San Juan']


In [50]:
merged_power_CAMS_dropped.describe()

,utc_time,Clear sky GHI,GHI,Solar_capacity,Cloud optical depth,Cloud coverage,Snow probability,Cloud type,daytime,loc_lat,...,generation other renewable,generation solar,generation waste,generation wind onshore,forecast solar day ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual
count,315576,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,...,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000,315576.000000
mean,2016-12-31 10:29:59.999999488,248.296706,215.047466,127.777778,2.657163,13.691435,0.746407,1.079569,0.506066,38.127513,...,85.634284,1433.124756,269.418475,5464.888034,1439.066735,5471.216689,28712.129962,28697.730083,49.874341,57.884023
min,2014-12-31 23:00:00,0.000000,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,36.660640,...,0.000000,0.000000,0.000000,0.000000,0.000000,237.000000,18105.000000,18041.000000,2.060000,9.330000
25%,2016-01-01 04:45:00,0.000000,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,37.431230,...,73.000000,71.000000,240.000000,2933.000000,69.000000,2979.000000,24793.750000,24807.000000,41.490000,49.347500
50%,2016-12-31 10:30:00,13.474000,10.486950,100.000000,0.000000,0.000000,0.000000,0.000000,0.583333,37.958970,...,88.000000,616.000000,279.000000,4849.000000,576.000000,4855.000000,28906.000000,28902.000000,50.520000,58.020000
75%,2017-12-31 16:15:00,498.478075,399.355025,150.000000,0.002754,1.250000,0.000000,0.000000,1.000000,39.184090,...,97.000000,2579.000000,310.000000,7398.000000,2636.000000,7353.000000,32263.250000,32193.250000,60.530000,68.010000
max,2018-12-31 22:00:00,1073.344200,1073.344000,200.000000,150.872500,100.000000,95.000000,8.000000,1.000000,39.239060,...,119.000000,5792.000000,357.000000,17436.000000,5836.000000,17430.000000,41390.000000,41015.000000,101.990000,116.800000
std,NaN,323.509586,298.415321,34.246799,10.667633,29.897269,5.479003,2.435486,0.485856,0.918539,...,14.076727,1680.420248,50.218767,3213.504423,1677.682089,3176.272592,4594.042622,4575.504114,14.618714,14.203903


In [25]:
with pd.option_context('display.max_columns', None):
    print(merged_power_CAMS_dropped.columns)

Index(['utc_time', 'Clear sky GHI', 'GHI', 'Solar_capacity',
       'Cloud optical depth', 'Cloud coverage', 'Snow probability',
       'Cloud type', 'daytime', 'loc_lat', 'loc_long', 'city_name', 'distance',
       'daytime_frac', 'daytime_any', 'generation biomass',
       'generation fossil brown coal/lignite', 'generation fossil gas',
       'generation fossil hard coal', 'generation fossil oil',
       'generation hydro pumped storage consumption',
       'generation hydro run-of-river and poundage',
       'generation hydro water reservoir', 'generation nuclear',
       'generation other', 'generation other renewable', 'generation solar',
       'generation waste', 'generation wind onshore',
       'forecast solar day ahead', 'forecast wind onshore day ahead',
       'total load forecast', 'total load actual', 'price day ahead',
       'price actual'],
      dtype='object')


In [26]:
# Create a weighting scheme for the various CAMS features as the following:
# CAMS_weight(i) = Solar_capacity(i)/sum[Solar_capacity(t)]
# Where i is for each location value for each row and t is for the sum across the locations at each time stamp.
# I won't worry about distances since the CAMS data is taken at the location of the solar plants and the distances are in reference
# to the nearest city.
weight_sum = merged_power_CAMS_dropped.groupby('utc_time')['Solar_capacity'].transform('sum')
merged_power_CAMS_dropped['CAMS_weight_norm'] = merged_power_CAMS_dropped['Solar_capacity']/weight_sum

# Weighted versions of CAMS variables
weighted_cols = ['GHI', 'Clear sky GHI', 'Cloud optical depth', 'Cloud coverage', 'Snow probability']

for col in weighted_cols:
        merged_power_CAMS_dropped[f'{col}_weighted'] = merged_power_CAMS_dropped[col] * merged_power_CAMS_dropped['CAMS_weight_norm']

In [27]:
merged_power_CAMS_dropped['ghi_clear_ratio_weighted'] = (
    merged_power_CAMS_dropped['GHI_weighted'] /
    merged_power_CAMS_dropped['Clear sky GHI_weighted']
)

merged_power_CAMS_dropped['ghi_clear_ratio_weighted'] = merged_power_CAMS_dropped['ghi_clear_ratio_weighted'].fillna(0)

merged_power_CAMS_dropped['GHI_x_cloud_coverage'] = (
    merged_power_CAMS_dropped['GHI_weighted'] *
    merged_power_CAMS_dropped['Cloud coverage_weighted']
)

merged_power_CAMS_dropped['GHI_x_cloud_coverage'] = merged_power_CAMS_dropped['GHI_x_cloud_coverage'].fillna(0)

merged_power_CAMS_dropped['GHI_x_optical_depth'] = (
    merged_power_CAMS_dropped['GHI_weighted'] *
    merged_power_CAMS_dropped['Cloud optical depth_weighted']
)

merged_power_CAMS_dropped['GHI_x_optical_depth'] = merged_power_CAMS_dropped['GHI_x_optical_depth'].fillna(0)

In [126]:
with pd.option_context('display.max_columns', None):
    print(merged_power_CAMS_dropped.describe())

                            utc_time  Clear sky GHI            GHI  \
count                         315576  315576.000000  315576.000000   
mean   2016-12-31 10:29:59.999999488     248.296706     215.047466   
min              2014-12-31 23:00:00       0.000000       0.000000   
25%              2016-01-01 04:45:00       0.000000       0.000000   
50%              2016-12-31 10:30:00      13.474000      10.486950   
75%              2017-12-31 16:15:00     498.478075     399.355025   
max              2018-12-31 22:00:00    1073.344200    1073.344000   
std                              NaN     323.509586     298.415321   

       Solar_capacity  Cloud optical depth  Cloud coverage  Snow probability  \
count   315576.000000        315576.000000   315576.000000     315576.000000   
mean       127.777778             2.657163       13.691435          0.746407   
min        100.000000             0.000000        0.000000          0.000000   
25%        100.000000             0.000000       

In [7]:
# Before applying weights to the feature columns, let's create some new features.
# Now I'll create a function to compute the Sun's elevation above the horizon.
import pvlib
from pvlib.location import Location
import pgeocode

def calculate_solar_position(df):
    times = df['utc_time']
    latitudes = df["loc_lat"]
    longitudes = df["loc_long"]
    solpos = pvlib.solarposition.get_solarposition(time=times, latitude=latitudes, longitude=longitudes)
    solar_elevation = solpos["apparent_elevation"].to_numpy()

    solar_elevation[solar_elevation < 0] = 0

    return solar_elevation     # Convert single element numpy array to float when returning

In [29]:
merged_power_CAMS_features = merged_power_CAMS_dropped.copy()
merged_power_CAMS_features['sol_elev'] = calculate_solar_position(merged_power_CAMS_features)

In [30]:
city_names = merged_power_CAMS_features['city_name'].unique()
city_coords = {}
for city in city_names:
    lat = merged_power_CAMS_features.loc[merged_power_CAMS_features['city_name'] == city, 'loc_lat'].values[0]
    lon = merged_power_CAMS_features.loc[merged_power_CAMS_features['city_name'] == city, 'loc_long'].values[0]
    city_coords[city] = (lat, lon)

print(city_coords)

{'Ecija': (37.57918, -5.15676), 'El Carpio': (37.95897, -4.50255), 'Guadix': (37.22853, -3.06854), 'Logrosan': (39.22472, -5.39056), 'San Jose del Valle': (36.66064, -5.8396), 'Santa Marta': (38.6412, -6.74474), 'Seville': (37.43123, -6.26031), 'Tomelloso': (39.18409, -3.31163), 'Villarta de San Juan': (39.23906, -3.4751)}


In [8]:
# Let's add the additional time/date features.
import holidays
from suntime import Sun
from datetime import date, datetime, timezone
from zoneinfo import ZoneInfo

# Compute the meteorological season based on month
def get_season(month):
    if month in [3, 4, 5]:
        # Spring
        return 1
    elif month in [6, 7, 8]:
        # Summer
        return 2
    elif month in [9, 10, 11]:
        # Autumn
        return 3
    else:
        # Winter
        return 0

# Determine the sunrise and sunset times.
def precompute_sunrise_sunset(city_coords_dict, all_dates):
    records = []
    for city, (lat, lon) in city_coords_dict.items():
        sun = Sun(lat, lon)
        for d in all_dates:
            # Make sure d is a datetime.date object (not pd.Timestamp)
            if isinstance(d, pd.Timestamp):
                d_py = d.date()
            else:
                d_py = d
            dt = datetime.combine(d_py, datetime.min.time()).replace(tzinfo=timezone.utc)
            try:
                sunrise = sun.get_sunrise_time(dt)
                sunset = sun.get_sunset_time(dt)
            except Exception as e:
                print(f"Failed for {city} {d_py}: {e}")
                sunrise, sunset = pd.NaT, pd.NaT
            records.append({'city_name': city, 'date': d_py, 'sunrise_time_utc': sunrise, 'sunset_time_utc': sunset})
    return pd.DataFrame(records)

def add_time_features(df, city_coords_dict, sun_table=None):
    df = df.copy()

    # Force utc_time to timezone-aware UTC datetimes
    df['utc_time'] = pd.to_datetime(df['utc_time'], utc=True)

    df['date'] = df['utc_time'].dt.date
    df['month'] = df['utc_time'].dt.month

    # If no precomputed table is given, build it now:
    if sun_table is None:
        all_dates = df['date'].unique()
        sun_table = precompute_sunrise_sunset(city_coords_dict, all_dates)

    # Merge sun_table (on city and date) into main dataframe
    df = df.merge(sun_table, on=['city_name', 'date'], how='left')

    # Remove timezone info for naive datetime columns (removes +00:00)
    df['sunrise_time_utc_naive'] = df['sunrise_time_utc'].dt.tz_localize(None)
    df['sunset_time_utc_naive'] = df['sunset_time_utc'].dt.tz_localize(None)

    # Daylight length (in hours)
    df['daylight_length_hr'] = (df['sunset_time_utc_naive'] - df['sunrise_time_utc_naive']).dt.total_seconds() / 3600.0

    neg_mask = df['daylight_length_hr'] < 0
    df.loc[neg_mask, 'daylight_length_hr'] = df.loc[neg_mask, 'daylight_length_hr'] + 24.0

    df['sunrise_time_utc'] = df['sunrise_time_utc_naive'].dt.time
    df['sunset_time_utc'] = df['sunset_time_utc_naive'].dt.time

    # Sunrise/sunset as fractions of day (UTC)
    df['sunrise_day_fraction_utc'] = (
        df['sunrise_time_utc_naive'].dt.hour / 24 +
        df['sunrise_time_utc_naive'].dt.minute / (24*60) +
        df['sunrise_time_utc_naive'].dt.second / (24*3600)
    )
    df['sunset_day_fraction_utc'] = (
        df['sunset_time_utc_naive'].dt.hour / 24 +
        df['sunset_time_utc_naive'].dt.minute / (24*60) +
        df['sunset_time_utc_naive'].dt.second / (24*3600)
    )

    df = df.drop(columns=['sunrise_time_utc_naive', 'sunset_time_utc_naive'])

    # UTC time as fraction of day
    df['utc_time_day_fraction'] = df['utc_time'].dt.hour/24.0 + df['utc_time'].dt.minute/(24.0*60) + df['utc_time'].dt.second/(24.0*3600)
    # Date as fraction of year
    df['day_of_year'] = df['utc_time'].dt.dayofyear
    df['date_fraction_of_year'] = df['day_of_year'] / 365.25

    # Day of week (Monday=0)
    df['day_of_week'] = df['utc_time'].dt.dayofweek
    # Is weekend (0 for Mon–Fri, 1 for Sat/Sun)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    # Cyclical time encodings
    df['day_hour_sin'] = np.sin(2 * np.pi * df['utc_time'].dt.hour / 24)
    df['day_hour_cos'] = np.cos(2 * np.pi * df['utc_time'].dt.hour / 24)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    df['season'] = df['month'].apply(get_season)

    spain_tz = ZoneInfo("Europe/Madrid")
    spain_time = df['utc_time'].dt.tz_convert(spain_tz)
    years = spain_time.dt.year.unique().tolist()
    #years = df['utc_time'].dt.year.unique().tolist()
    es_holidays = holidays.country_holidays('ES', years=years)
    #  Is holiday (1 for holiday, 0 for not a holiday)
    # df['is_holiday'] = df['utc_time'].dt.date.isin(es_holidays).astype(int)
    df['is_holiday'] = spain_time.dt.date.isin(es_holidays).astype(int)

    # Peak usage time (if not a holiday and not the weekend and df['utc_time'].dt.hour >= 10 and df['utc_time'].dt.hour <= 16).
    # df['is_peak'] = ((df['is_holiday'] == 0) &
    #                  (df['is_weekend'] == 0) &
    #                  (df['utc_time'].dt.hour.between(9, 17))
    #                 ).astype(int)

    df['is_peak'] = ((df['is_holiday'] == 0) &
                     (df['is_weekend'] == 0) &
                     (spain_time.dt.hour.between(9, 17))
                    ).astype(int)

    # Remove timezone info for naive datetime columns (removes +00:00)
    df['utc_time'] = df['utc_time'].dt.tz_localize(None)

    return df

In [32]:
merged_power_CAMS_features2 = merged_power_CAMS_features.copy(deep=True)

# Precompute sunrise/sunset for all unique dates and cities
all_dates = merged_power_CAMS_features2['utc_time'].dt.date.unique()
sun_table = precompute_sunrise_sunset(city_coords, all_dates)

merged_power_CAMS_features2 = add_time_features(merged_power_CAMS_features2, city_coords, sun_table)

direct_out = current_dir + '/output'
filename_out = direct_out + '/power_CAMS_dataset_with_new_features.csv'
merged_power_CAMS_features2.to_csv(filename_out, index=False)

In [33]:
with pd.option_context('display.max_columns', None):
    print(merged_power_CAMS_features2.columns)

Index(['utc_time', 'Clear sky GHI', 'GHI', 'Solar_capacity',
       'Cloud optical depth', 'Cloud coverage', 'Snow probability',
       'Cloud type', 'daytime', 'loc_lat', 'loc_long', 'city_name', 'distance',
       'daytime_frac', 'daytime_any', 'generation biomass',
       'generation fossil brown coal/lignite', 'generation fossil gas',
       'generation fossil hard coal', 'generation fossil oil',
       'generation hydro pumped storage consumption',
       'generation hydro run-of-river and poundage',
       'generation hydro water reservoir', 'generation nuclear',
       'generation other', 'generation other renewable', 'generation solar',
       'generation waste', 'generation wind onshore',
       'forecast solar day ahead', 'forecast wind onshore day ahead',
       'total load forecast', 'total load actual', 'price day ahead',
       'price actual', 'CAMS_weight_norm', 'GHI_weighted',
       'Clear sky GHI_weighted', 'Cloud optical depth_weighted',
       'Cloud coverage_weight

In [34]:
merged_power_CAMS_features_final = merged_power_CAMS_features2.copy(deep=True)

# Weather columns to roll by city
weather_features = ['GHI', 'Cloud optical depth', 'Cloud coverage', 'Snow probability', 'Cloud type', 'GHI_weighted',
                    'Clear sky GHI_weighted', 'Cloud optical depth_weighted', 'Cloud coverage_weighted',
                    'Snow probability_weighted', 'ghi_clear_ratio_weighted', 'GHI_x_cloud_coverage', 'GHI_x_optical_depth']

# Power features to roll (national)
power_features = [
    'generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas',
    'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage', 'generation hydro water reservoir',
    'generation nuclear', 'generation other', 'generation other renewable', 'generation solar',
    'generation waste', 'generation wind onshore'
]

# Features to roll for both 3h and 6h (e.g., for load and price)
special_features = ['total load actual', 'price actual']

In [ ]:
# Create rolling and lag features
# Rolling mean given in hours.
roll_by = [3, 6]
# Lag by the given in hours.
lag_by = [1, 2, 3, 24]

# WEATHER: by city_name and sorted by utc_time
merged_power_CAMS_features_final = merged_power_CAMS_features_final.sort_values(['city_name', 'utc_time']).reset_index(drop=True)

for col in weather_features:

    # First value of this column within each city
    first_value_per_city = merged_power_CAMS_features_final.groupby('city_name')[col].transform('first')

    for roll in roll_by:
        roll_col = f'{col}_roll{roll}h'

        merged_power_CAMS_features_final[roll_col] = (
            merged_power_CAMS_features_final
            .groupby('city_name')[col]
            .transform(lambda x: x.shift(1).rolling(window=roll, min_periods=1).mean())
        )

        # Fill NaN in the first row of each city with the first row value for that city
        merged_power_CAMS_features_final[roll_col] = (
            merged_power_CAMS_features_final[roll_col]
            .fillna(first_value_per_city)
        )

    for lag in lag_by:
        lag_col = f'{col}_lag{lag}h'

        merged_power_CAMS_features_final[lag_col] = (
            merged_power_CAMS_features_final
            .groupby('city_name')[col]
            .transform(lambda x: x.shift(lag))
        )

        # Fill first `lag` rows of each city with the first row value for that city
        merged_power_CAMS_features_final[lag_col] = (
            merged_power_CAMS_features_final[lag_col]
            .fillna(first_value_per_city)
        )

# POWER: National level still needs to be handled on city level due to repeated values
#merged_power_CAMS_features_final = merged_power_CAMS_features_final.sort_values('utc_time')
for col in power_features:

    # First value of this column within each city
    first_value_per_city = merged_power_CAMS_features_final.groupby('city_name')[col].transform('first')

    for roll in roll_by:
        roll_col = f'{col}_roll{roll}h'

        merged_power_CAMS_features_final[roll_col] = (
            merged_power_CAMS_features_final
            .groupby('city_name')[col]
            .transform(lambda x: x.shift(1).rolling(window=roll, min_periods=1).mean())
        )

        # Fill NaN in the first row of each city with the first row value for that city
        merged_power_CAMS_features_final[roll_col] = (
            merged_power_CAMS_features_final[roll_col]
            .fillna(first_value_per_city)
        )

    for lag in lag_by:
        lag_col = f'{col}_lag{lag}h'

        merged_power_CAMS_features_final[lag_col] = (
            merged_power_CAMS_features_final
            .groupby('city_name')[col]
            .transform(lambda x: x.shift(lag))
        )

        # Fill first `lag` rows of each city with the first row value for that city
        merged_power_CAMS_features_final[lag_col] = (
            merged_power_CAMS_features_final[lag_col]
            .fillna(first_value_per_city)
        )

# Special features (total load actual, price actual): 3h and 6h
for col in special_features:

    # First value of this column within each city
    first_value_per_city = merged_power_CAMS_features_final.groupby('city_name')[col].transform('first')

    for roll in roll_by:
        roll_col = f'{col}_roll{roll}h'

        merged_power_CAMS_features_final[roll_col] = (
            merged_power_CAMS_features_final
            .groupby('city_name')[col]
            .transform(lambda x: x.shift(1).rolling(window=roll, min_periods=1).mean())
        )

        # Fill NaN in the first row of each city with the first row value for that city
        merged_power_CAMS_features_final[roll_col] = (
            merged_power_CAMS_features_final[roll_col]
            .fillna(first_value_per_city)
        )

    for lag in lag_by:
        lag_col = f'{col}_lag{lag}h'

        merged_power_CAMS_features_final[lag_col] = (
            merged_power_CAMS_features_final
            .groupby('city_name')[col]
            .transform(lambda x: x.shift(lag))
        )

        # Fill first `lag` rows of each city with the first row value for that city
        merged_power_CAMS_features_final[lag_col] = (
            merged_power_CAMS_features_final[lag_col]
            .fillna(first_value_per_city)
        )

merged_power_CAMS_features_final = merged_power_CAMS_features_final.sort_values('utc_time').reset_index(drop=True)

In [ ]:
print('utc_time', ',', 'daytime_frac', ',', 'daytime_any', ',', 'sol_elev',
      ',', 'sunrise_time_utc', ',', 'sunset_time_utc', ',', 'GHI', ',', 'GHI_roll3h', ',', 'GHI_lag3h',
      ',', 'solar', ',', 'solar_roll3h', ',', 'solar_lag3h', ',', 'city_name')
for i in range(200):
    print(merged_power_CAMS_features_final.loc[i, 'utc_time'], ',',
          round(merged_power_CAMS_features_final.loc[i, 'daytime_frac'], 2), ',',
          merged_power_CAMS_features_final.loc[i, 'daytime_any'], ',',
          round(merged_power_CAMS_features_final.loc[i, 'sol_elev'], 2), ',',
          merged_power_CAMS_features_final.loc[i, 'sunrise_time_utc'], ',',
          merged_power_CAMS_features_final.loc[i, 'sunset_time_utc'], ',',
          round(merged_power_CAMS_features_final.loc[i, 'GHI'], 2), ',',
          round(merged_power_CAMS_features_final.loc[i, 'GHI_roll3h'], 2), ',',
          round(merged_power_CAMS_features_final.loc[i, 'GHI_lag3h'], 2), ',',
          round(merged_power_CAMS_features_final.loc[i, 'generation solar'], 2), ',',
          round(merged_power_CAMS_features_final.loc[i, 'generation solar_roll3h'], 2), ',',
          round(merged_power_CAMS_features_final.loc[i, 'generation solar_lag3h'], 2), ',',
          merged_power_CAMS_features_final.loc[i, 'city_name'])

In [36]:
# Let's clean up the dataframe a bit drop some duplicate columns

merged_power_CAMS_features_final = merged_power_CAMS_features_final.drop(columns=['daytime'])

In [37]:
with pd.option_context('display.max_columns', None):
    print(list(merged_power_CAMS_features_final.columns))

['utc_time', 'Clear sky GHI', 'GHI', 'Solar_capacity', 'Cloud optical depth', 'Cloud coverage', 'Snow probability', 'Cloud type', 'loc_lat', 'loc_long', 'city_name', 'distance', 'daytime_frac', 'daytime_any', 'generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas', 'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption', 'generation hydro run-of-river and poundage', 'generation hydro water reservoir', 'generation nuclear', 'generation other', 'generation other renewable', 'generation solar', 'generation waste', 'generation wind onshore', 'forecast solar day ahead', 'forecast wind onshore day ahead', 'total load forecast', 'total load actual', 'price day ahead', 'price actual', 'CAMS_weight_norm', 'GHI_weighted', 'Clear sky GHI_weighted', 'Cloud optical depth_weighted', 'Cloud coverage_weighted', 'Snow probability_weighted', 'ghi_clear_ratio_weighted', 'GHI_x_cloud_coverage', 'GHI_x_optical_depth', 'sol_elev', '

In [38]:
# Now let's prepare dataframe for solar power prediction by removing all of the unwanted power columns.
merged_CAMS_features_solar = merged_power_CAMS_features_final.copy(deep=True)

power_columns_remove = ['generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas',
                        'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption',
                        'generation hydro run-of-river and poundage', 'generation hydro water reservoir', 'generation nuclear',
                        'generation other', 'generation other renewable', 'generation waste', 'generation wind onshore',
                        'forecast wind onshore day ahead', 'total load forecast', 'total load actual', 'price day ahead',
                        'price actual']

# Find all columns to drop:
# - exact base column names
# - any derived columns such as _lag1h, _roll3h, etc.
cols_to_drop = [
    col for col in merged_CAMS_features_solar.columns
    if any(col == base_col or col.startswith(base_col + '_') for base_col in power_columns_remove)
]

merged_CAMS_features_solar = merged_CAMS_features_solar.drop(columns=cols_to_drop)

# Let's also drop the distance column as it won't be useful here (distance is to nearest city as given in city_name but
# the actual CAMS data comes from the location of the solar plant, not the city. Additionally, drop 'sunrise_time_utc'
# and 'sunset_time_utc' as these are already encoding in 'sunrise_day_fraction_utc' and 'sunset_day_fraction_utc'.
merged_CAMS_features_solar = merged_CAMS_features_solar.drop(columns=['distance', 'sunrise_day_fraction_utc',
                                                                      'sunset_day_fraction_utc'])

cols_to_drop.extend(['distance', 'sunrise_day_fraction_utc', 'sunset_day_fraction_utc'])

print(f"Number of columns dropped: {len(cols_to_drop)}")
print("Dropped columns:")
print(cols_to_drop)

Number of columns dropped: 111
Dropped columns:
['generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas', 'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption', 'generation hydro run-of-river and poundage', 'generation hydro water reservoir', 'generation nuclear', 'generation other', 'generation other renewable', 'generation waste', 'generation wind onshore', 'forecast wind onshore day ahead', 'total load forecast', 'total load actual', 'price day ahead', 'price actual', 'generation biomass_roll3h', 'generation biomass_roll6h', 'generation biomass_lag1h', 'generation biomass_lag2h', 'generation biomass_lag3h', 'generation biomass_lag24h', 'generation fossil brown coal/lignite_roll3h', 'generation fossil brown coal/lignite_roll6h', 'generation fossil brown coal/lignite_lag1h', 'generation fossil brown coal/lignite_lag2h', 'generation fossil brown coal/lignite_lag3h', 'generation fossil brown coal/lignite_lag24

In [158]:
with pd.option_context('display.max_columns', None):
    print(list(merged_CAMS_features_solar.columns))

['utc_time', 'Clear sky GHI', 'GHI', 'Solar_capacity', 'Cloud optical depth', 'Cloud coverage', 'Snow probability', 'Cloud type', 'loc_lat', 'loc_long', 'city_name', 'daytime_frac', 'daytime_any', 'generation solar', 'forecast solar day ahead', 'CAMS_weight_norm', 'GHI_weighted', 'Clear sky GHI_weighted', 'Cloud optical depth_weighted', 'Cloud coverage_weighted', 'Snow probability_weighted', 'ghi_clear_ratio_weighted', 'GHI_x_cloud_coverage', 'GHI_x_optical_depth', 'sol_elev', 'date', 'month', 'sunrise_time_utc', 'sunset_time_utc', 'daylight_length_hr', 'utc_time_day_fraction', 'day_of_year', 'date_fraction_of_year', 'day_of_week', 'is_weekend', 'day_hour_sin', 'day_hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos', 'month_sin', 'month_cos', 'season', 'is_holiday', 'is_peak', 'GHI_roll3h', 'GHI_roll6h', 'GHI_lag1h', 'GHI_lag2h', 'GHI_lag3h', 'GHI_lag24h', 'Cloud optical depth_roll3h', 'Cloud optical depth_roll6h', 'Cloud optical depth_lag1h', 'Cloud 

In [9]:
# I'll start with predicting solar generation.

def aggregate_solar_to_national(
    df,
    datetime_col='utc_time',
    city_col='city_name',
    weight_col='CAMS_weight_norm',
    target_col='generation solar',
    forecast_col='forecast solar day ahead'
):
    """
    Aggregate city-level CAMS + solar features to one national row per timestamp.

    Assumes:
    - one or more rows per timestamp, one per city
    - target_col is duplicated across cities at each timestamp
    - weight_col is already normalized within timestamp, or close to it
    """

    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    df = df.sort_values([datetime_col, city_col]).reset_index(drop=True)

    # -----------------------------
    # Column groups
    # -----------------------------

    # Already weighted contribution-style features: sum across cities
    weighted_sum_cols = [
        'GHI_weighted',
        'Clear sky GHI_weighted',
        'Cloud optical depth_weighted',
        'Cloud coverage_weighted',
        'Snow probability_weighted',
        'ghi_clear_ratio_weighted',
        'GHI_x_cloud_coverage',
        'GHI_x_optical_depth',

        'GHI_weighted_roll3h',
        'GHI_weighted_roll6h',
        'GHI_weighted_lag1h',
        'GHI_weighted_lag2h',
        'GHI_weighted_lag3h',
        'GHI_weighted_lag24h',

        'Clear sky GHI_weighted_roll3h',
        'Clear sky GHI_weighted_roll6h',
        'Clear sky GHI_weighted_lag1h',
        'Clear sky GHI_weighted_lag2h',
        'Clear sky GHI_weighted_lag3h',
        'Clear sky GHI_weighted_lag24h',

        'Cloud optical depth_weighted_roll3h',
        'Cloud optical depth_weighted_roll6h',
        'Cloud optical depth_weighted_lag1h',
        'Cloud optical depth_weighted_lag2h',
        'Cloud optical depth_weighted_lag3h',
        'Cloud optical depth_weighted_lag24h',

        'Cloud coverage_weighted_roll3h',
        'Cloud coverage_weighted_roll6h',
        'Cloud coverage_weighted_lag1h',
        'Cloud coverage_weighted_lag2h',
        'Cloud coverage_weighted_lag3h',
        'Cloud coverage_weighted_lag24h',

        'Snow probability_weighted_roll3h',
        'Snow probability_weighted_roll6h',
        'Snow probability_weighted_lag1h',
        'Snow probability_weighted_lag2h',
        'Snow probability_weighted_lag3h',
        'Snow probability_weighted_lag24h',

        'ghi_clear_ratio_weighted_roll3h',
        'ghi_clear_ratio_weighted_roll6h',
        'ghi_clear_ratio_weighted_lag1h',
        'ghi_clear_ratio_weighted_lag2h',
        'ghi_clear_ratio_weighted_lag3h',
        'ghi_clear_ratio_weighted_lag24h',

        'GHI_x_cloud_coverage_roll3h',
        'GHI_x_cloud_coverage_roll6h',
        'GHI_x_cloud_coverage_lag1h',
        'GHI_x_cloud_coverage_lag2h',
        'GHI_x_cloud_coverage_lag3h',
        'GHI_x_cloud_coverage_lag24h',

        'GHI_x_optical_depth_roll3h',
        'GHI_x_optical_depth_roll6h',
        'GHI_x_optical_depth_lag1h',
        'GHI_x_optical_depth_lag2h',
        'GHI_x_optical_depth_lag3h',
        'GHI_x_optical_depth_lag24h',
    ]

    # Raw city-level features: aggregate as weighted means using CAMS_weight_norm
    weighted_mean_cols = [
        'Clear sky GHI',
        'GHI',
        'Cloud optical depth',
        'Cloud coverage',
        'Snow probability',
        'Cloud type',
        'distance',
        'daytime_frac',
        'sol_elev',
        'daylight_length_hr',
        'sunrise_day_fraction_utc',
        'sunset_day_fraction_utc',
        'daytime_any',

        'GHI_roll3h',
        'GHI_roll6h',
        'GHI_lag1h',
        'GHI_lag2h',
        'GHI_lag3h',
        'GHI_lag24h',

        'Cloud optical depth_roll3h',
        'Cloud optical depth_roll6h',
        'Cloud optical depth_lag1h',
        'Cloud optical depth_lag2h',
        'Cloud optical depth_lag3h',
        'Cloud optical depth_lag24h',

        'Cloud coverage_roll3h',
        'Cloud coverage_roll6h',
        'Cloud coverage_lag1h',
        'Cloud coverage_lag2h',
        'Cloud coverage_lag3h',
        'Cloud coverage_lag24h',

        'Snow probability_roll3h',
        'Snow probability_roll6h',
        'Snow probability_lag1h',
        'Snow probability_lag2h',
        'Snow probability_lag3h',
        'Snow probability_lag24h',

        'Cloud type_roll3h',
        'Cloud type_roll6h',
        'Cloud type_lag1h',
        'Cloud type_lag2h',
        'Cloud type_lag3h',
        'Cloud type_lag24h',
    ]

    # Same for every city at each timestamp: use first
    first_cols = [
        target_col,
        forecast_col,
        'date',
        'month',
        'utc_time_day_fraction',
        'day_of_year',
        'date_fraction_of_year',
        'day_of_week',
        'is_weekend',
        'day_hour_sin',
        'day_hour_cos',
        'day_of_week_sin',
        'day_of_week_cos',
        'day_of_year_sin',
        'day_of_year_cos',
        'month_sin',
        'month_cos',
        'season',
        'is_holiday',
        'is_peak',
        'generation solar_roll3h',
        'generation solar_roll6h',
        'generation solar_lag1h',
        'generation solar_lag2h',
        'generation solar_lag3h',
        'generation solar_lag24h',
    ]

    # Sum-type metadata
    sum_cols = [
        'Solar_capacity'
    ]

    # Max/bool-like columns
    max_cols = []

    # Optional count of cities contributing
    extra_cols = [
        city_col, weight_col
    ]

    # Keep only columns that actually exist
    weighted_sum_cols = [c for c in weighted_sum_cols if c in df.columns]
    weighted_mean_cols = [c for c in weighted_mean_cols if c in df.columns]
    first_cols = [c for c in first_cols if c in df.columns]
    sum_cols = [c for c in sum_cols if c in df.columns]
    max_cols = [c for c in max_cols if c in df.columns]

    # -----------------------------
    # Helper for weighted mean
    # -----------------------------
    def weighted_mean(group, value_col, w_col):
        vals = group[value_col]
        w = group[w_col]

        mask = vals.notna() & w.notna()
        if not mask.any():
            return np.nan

        vals = vals[mask]
        w = w[mask]

        w_sum = w.sum()
        if w_sum == 0:
            return vals.mean()

        return np.sum(vals * w) / w_sum

    # -----------------------------
    # Aggregate
    # -----------------------------
    grouped = df.groupby(datetime_col, sort=True)

    # First / sum / max blocks
    agg_parts = []

    if first_cols:
        agg_first = grouped[first_cols].first()
        agg_parts.append(agg_first)

    if sum_cols:
        agg_sum = grouped[sum_cols].sum()
        agg_parts.append(agg_sum)

    if max_cols:
        agg_max = grouped[max_cols].max()
        agg_parts.append(agg_max)

    if city_col in df.columns:
        city_count = grouped[city_col].nunique().rename('n_cities')
        agg_parts.append(city_count)

    if weight_col in df.columns:
        weight_sum = grouped[weight_col].sum().rename('weight_sum_check')
        agg_parts.append(weight_sum)

    # Sum already-weighted features
    if weighted_sum_cols:
        agg_weighted_sum = grouped[weighted_sum_cols].sum()
        agg_parts.append(agg_weighted_sum)

    # Weighted-mean raw features
    if weighted_mean_cols:
        weighted_mean_frames = []

        for col in weighted_mean_cols:
            temp = df[[datetime_col, col, weight_col]].copy()
    
            mask = temp[col].notna() & temp[weight_col].notna()
            temp = temp.loc[mask].copy()
    
            temp['_wx'] = temp[col] * temp[weight_col]
    
            agg_temp = temp.groupby(datetime_col).agg(
                wx_sum=('_wx', 'sum'),
                w_sum=(weight_col, 'sum')
            )
    
            agg_temp[col] = agg_temp['wx_sum'] / agg_temp['w_sum'].replace(0, np.nan)
    
            weighted_mean_frames.append(agg_temp[[col]])
    
        if weighted_mean_frames:
            agg_weighted_mean = pd.concat(weighted_mean_frames, axis=1)
            agg_parts.append(agg_weighted_mean)

    # Combine all pieces
    national_df = pd.concat(agg_parts, axis=1).reset_index()

    # Sort
    national_df = national_df.sort_values(datetime_col).reset_index(drop=True)

    return national_df

In [40]:
national_solar_df = aggregate_solar_to_national(
    merged_CAMS_features_solar,
    datetime_col='utc_time',
    city_col='city_name',
    weight_col='CAMS_weight_norm',
    target_col='generation solar',
    forecast_col='forecast solar day ahead'
)

print(national_solar_df.shape)
print(national_solar_df.columns.tolist())

(35064, 126)
['utc_time', 'generation solar', 'forecast solar day ahead', 'date', 'month', 'utc_time_day_fraction', 'day_of_year', 'date_fraction_of_year', 'day_of_week', 'is_weekend', 'day_hour_sin', 'day_hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos', 'month_sin', 'month_cos', 'season', 'is_holiday', 'is_peak', 'generation solar_roll3h', 'generation solar_roll6h', 'generation solar_lag1h', 'generation solar_lag2h', 'generation solar_lag3h', 'generation solar_lag24h', 'Solar_capacity', 'n_cities', 'weight_sum_check', 'GHI_weighted', 'Clear sky GHI_weighted', 'Cloud optical depth_weighted', 'Cloud coverage_weighted', 'Snow probability_weighted', 'ghi_clear_ratio_weighted', 'GHI_x_cloud_coverage', 'GHI_x_optical_depth', 'GHI_weighted_roll3h', 'GHI_weighted_roll6h', 'GHI_weighted_lag1h', 'GHI_weighted_lag2h', 'GHI_weighted_lag3h', 'GHI_weighted_lag24h', 'Clear sky GHI_weighted_roll3h', 'Clear sky GHI_weighted_roll6h', 'Clear sky GHI_weighted_lag1h',

In [41]:
direct_out = current_dir + '/output'
filename_out = direct_out + '/agreggated_CAMS_solar_dataset_column_stats.csv'
national_solar_df.describe().to_csv(filename_out, index=True)
filename_out = direct_out + '/agreggated_CAMS_solar_dataset.csv'
national_solar_df.to_csv(filename_out, index=True)

In [10]:
def bokeh_plot_seasonal_decompose(result, title_prefix=""):
    """
    Interactive Bokeh plot of statsmodels seasonal_decompose results.
    
    Parameters:
    -----------
    result : statsmodels.tsa.seasonal.DecomposeResult
        Output of seasonal_decompose.
    title_prefix : str
        Prefix for subplot titles.

    Returns:
    --------
    plot_object : bokeh.layouts.column
        The Bokeh column layout object.
    """
    components = ['observed', 'trend', 'seasonal', 'resid']
    tooltips = [("Time", "@x{%F %H:%M}"), ("Value", "@y{0.00}")]
    formatter = {"@x": "datetime"}

    plots = []
    for comp in components:
        y = getattr(result, comp)
        p = figure(height=250, width=1200, 
                   x_axis_type='datetime',
                   tools="pan,wheel_zoom,box_zoom,reset,save",
                   title=f"{title_prefix} {comp.capitalize()}")
        p.line(y.index, y.values, line_width=2, color="royalblue")
        p.add_tools(HoverTool(
            tooltips=tooltips, formatters=formatter, mode='vline'
        ))
        p.xaxis.formatter = DatetimeTickFormatter(
            days="%d-%m-%Y",
            months="%b %Y",
            years="%Y",
            hours="%d-%m-%Y %H:%M"
        )
        p.yaxis.axis_label = comp.capitalize()
        plots.append(p)
    
    # Only show x-axis labels for bottom plot to reduce clutter
    # for plot in plots[:-1]:
    #     plot.xaxis.visible = False

    plot_object = column(*plots)

    return plot_object

In [64]:
datetime_col = 'utc_time'
target_col = 'generation solar'
forecast_col = 'forecast solar day ahead'
target_related_col = ['generation solar_roll3h', 'generation solar_roll6h', 'generation solar_lag1h', 'generation solar_lag2h',
                      'generation solar_lag3h', 'generation solar_lag24h']

features_cols = ['month', 'utc_time_day_fraction', 'day_of_year', 'date_fraction_of_year', 'day_of_week', 'is_weekend',
                 'day_hour_sin', 'day_hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos',
                 'month_sin', 'month_cos', 'season', 'is_holiday', 'is_peak', 'Solar_capacity', 'GHI_weighted', 'Clear sky GHI_weighted',
                 'Cloud optical depth_weighted', 'Cloud coverage_weighted', 'Snow probability_weighted', 'ghi_clear_ratio_weighted',
                 'GHI_x_cloud_coverage', 'GHI_x_optical_depth', 'GHI_weighted_roll3h', 'GHI_weighted_roll6h', 'GHI_weighted_lag1h',
                 'GHI_weighted_lag2h', 'GHI_weighted_lag3h', 'GHI_weighted_lag24h', 'Clear sky GHI_weighted_roll3h',
                 'Clear sky GHI_weighted_roll6h', 'Clear sky GHI_weighted_lag1h', 'Clear sky GHI_weighted_lag2h',
                 'Clear sky GHI_weighted_lag3h', 'Clear sky GHI_weighted_lag24h', 'Cloud optical depth_weighted_roll3h',
                 'Cloud optical depth_weighted_roll6h', 'Cloud optical depth_weighted_lag1h', 'Cloud optical depth_weighted_lag2h',
                 'Cloud optical depth_weighted_lag3h', 'Cloud optical depth_weighted_lag24h', 'Cloud coverage_weighted_roll3h',
                 'Cloud coverage_weighted_roll6h', 'Cloud coverage_weighted_lag1h', 'Cloud coverage_weighted_lag2h',
                 'Cloud coverage_weighted_lag3h', 'Cloud coverage_weighted_lag24h', 'Snow probability_weighted_roll3h',
                 'Snow probability_weighted_roll6h', 'Snow probability_weighted_lag1h', 'Snow probability_weighted_lag2h',
                 'Snow probability_weighted_lag3h', 'Snow probability_weighted_lag24h', 'ghi_clear_ratio_weighted_roll3h',
                 'ghi_clear_ratio_weighted_roll6h', 'ghi_clear_ratio_weighted_lag1h', 'ghi_clear_ratio_weighted_lag2h',
                 'ghi_clear_ratio_weighted_lag3h', 'ghi_clear_ratio_weighted_lag24h', 'GHI_x_cloud_coverage_roll3h',
                 'GHI_x_cloud_coverage_roll6h', 'GHI_x_cloud_coverage_lag1h', 'GHI_x_cloud_coverage_lag2h',
                 'GHI_x_cloud_coverage_lag3h', 'GHI_x_cloud_coverage_lag24h', 'GHI_x_optical_depth_roll3h',
                 'GHI_x_optical_depth_roll6h', 'GHI_x_optical_depth_lag1h', 'GHI_x_optical_depth_lag2h', 'GHI_x_optical_depth_lag3h',
                 'GHI_x_optical_depth_lag24h', 'Clear sky GHI', 'GHI', 'Cloud optical depth', 'Cloud coverage', 'Snow probability',
                 'Cloud type', 'daytime_frac', 'sol_elev', 'daylight_length_hr', 'daytime_any', 'GHI_roll3h', 'GHI_roll6h',
                 'GHI_lag1h', 'GHI_lag2h', 'GHI_lag3h', 'GHI_lag24h', 'Cloud optical depth_roll3h', 'Cloud optical depth_roll6h',
                 'Cloud optical depth_lag1h', 'Cloud optical depth_lag2h', 'Cloud optical depth_lag3h', 'Cloud optical depth_lag24h',
                 'Cloud coverage_roll3h', 'Cloud coverage_roll6h', 'Cloud coverage_lag1h', 'Cloud coverage_lag2h',
                 'Cloud coverage_lag3h', 'Cloud coverage_lag24h', 'Snow probability_roll3h', 'Snow probability_roll6h',
                 'Snow probability_lag1h', 'Snow probability_lag2h', 'Snow probability_lag3h', 'Snow probability_lag24h',
                 'Cloud type_roll3h', 'Cloud type_roll6h', 'Cloud type_lag1h', 'Cloud type_lag2h', 'Cloud type_lag3h',
                 'Cloud type_lag24h']

# Create two additional dataframes that will be used to split into training and testing sets. data contains all the CAMS features
# plus time features but does not contain the rolling and lag features for solar generation while data_lag_roll_solar
# contains these features. I will then fit two ML models, one without and one with the solar generation lag and roll features.
# The solar roll and lag features will need to be handled recursively when making model predictions using the predicted solar
# generation power to avoid data leakage.
feature_cols_model = features_cols
feature_cols_w_solar_roll_lag = features_cols + target_related_col

data = national_solar_df[[target_col, forecast_col, datetime_col] + feature_cols_model].copy()
data_lag_roll_solar = national_solar_df[[target_col, forecast_col, datetime_col] + feature_cols_w_solar_roll_lag].copy()

data = data.sort_values(datetime_col).reset_index(drop=True)
data = data.set_index(datetime_col).asfreq('h').reset_index()
#data = data.set_index(datetime_col).asfreq('h')
data = data.dropna(subset=[target_col] + feature_cols_model).copy()

data_lag_roll_solar = data_lag_roll_solar.sort_values(datetime_col).reset_index(drop=True)
data_lag_roll_solar = data_lag_roll_solar.set_index(datetime_col).asfreq('h').reset_index()
#data_lag_roll_solar = data_lag_roll_solar.set_index(datetime_col).asfreq('h')
data_lag_roll_solar = data_lag_roll_solar.dropna(subset=[target_col] + feature_cols_w_solar_roll_lag).copy()

N = len(data)

#------- split the data into a 70/30 training and testing set. Model 1 -- No solar generation roll or lag features --------------
split_idx = int(N * 0.7)

train = data.iloc[:split_idx].copy()
test = data.iloc[split_idx:].copy()

cat_cols = [c for c in ['month', 'day_of_week', 'season'] if c in feature_cols_model]
for c in cat_cols:
    train[c] = train[c].astype('category')
    test[c] = test[c].astype('category')

train = train.set_index('utc_time', drop=False).asfreq('h')
test = test.set_index('utc_time', drop=False).asfreq('h')

# Split into X/y
y_train = train[target_col]
X_train = train[feature_cols_model]
y_test = test[target_col]
X_test = test[feature_cols_model]
forecast_test = test[forecast_col]
datetime_test = test[datetime_col].to_numpy(copy=True)  # Save for plotting

In [65]:
print(test.index.values)

['2017-10-19T15:00:00.000000000' '2017-10-19T16:00:00.000000000'
 '2017-10-19T17:00:00.000000000' ... '2018-12-31T20:00:00.000000000'
 '2018-12-31T21:00:00.000000000' '2018-12-31T22:00:00.000000000']


In [66]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- 2) Make a time-ordered validation split from the training period (no shuffling)
val_frac = 0.2
val_cut = int(len(X_train) * (1 - val_frac))
X_tr, y_tr = X_train.iloc[:val_cut], y_train.iloc[:val_cut]
X_val, y_val = X_train.iloc[val_cut:], y_train.iloc[val_cut:]

fit_set = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols or None, free_raw_data=False)
val_set = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols or None, free_raw_data=False)

# --- 3) Train LightGBM with early stopping. This is the mean model.
mean_params = dict(
    objective='regression',
    boosting_type='gbdt',
    metric='rmse',
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    min_data_in_leaf=50,
    bagging_freq=1,
    reg_alpha=0.0,
    reg_lambda=0.0,
    seed=42,
    verbosity=-1,
    n_jobs=-1
)

mean_model = lgb.train(
    params=mean_params,
    train_set=fit_set,
    num_boost_round=5000,                # generous upper bound
    valid_sets=[fit_set, val_set],
    valid_names=["train","val"],
    callbacks=[lgb.early_stopping(stopping_rounds=200), lgb.log_evaluation(200)]
)

best_iter_mean = mean_model.best_iteration
print(f"Mean model best_iteration = {best_iter_mean}")

Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 287.629	val's rmse: 644.502
Early stopping, best iteration is:
[92]	train's rmse: 419.976	val's rmse: 630.749
Mean model best_iteration = 92


In [11]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------
# --- 4) Train quantile models for lower/upper bounds

def train_quantile(alpha, best_rounds):
    q_params = dict(
        objective="quantile",
        boosting_type='gbdt',
        alpha=alpha,
        metric="quantile",
        learning_rate=0.05,
        num_leaves=64,
        max_depth=-1,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        min_data_in_leaf=50,
        bagging_freq=1,
        reg_alpha=0.0,
        reg_lambda=0.0,
        seed=42,
        verbosity=-1,
        n_jobs=-1
    )
    
    # Use best_iter_mean as an upper bound; still allow early stopping to stop earlier
    model_q = lgb.train(
        params=q_params,
        train_set=fit_set,
        num_boost_round=best_rounds,
        valid_sets=[fit_set, val_set],
        valid_names=["train","val"],
        callbacks=[lgb.early_stopping(max(50, best_rounds // 10)), lgb.log_evaluation(max(50, best_rounds // 10))]
    )
    return model_q

In [ ]:
# 16th percentile (lower ~ 1-sigma)
lgbm_lower_model = train_quantile(alpha=0.16, best_rounds=best_iter_mean)

# Median prediction
lgbm_median_model = train_quantile(alpha=0.50, best_rounds=best_iter_mean)  # "median" model

# 84th percentile (upper ~ 1-sigma)
lgbm_upper_model = train_quantile(alpha=0.84, best_rounds=best_iter_mean)

In [68]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------

# --- 5) Predict on the test set
y_pred_mean = mean_model.predict(X_test, num_iteration=mean_model.best_iteration)
y_pred_lower  = lgbm_lower_model.predict(X_test,  num_iteration=lgbm_lower_model.best_iteration)
y_pred_median  = lgbm_median_model.predict(X_test,  num_iteration=lgbm_median_model.best_iteration)
y_pred_upper  = lgbm_upper_model.predict(X_test,  num_iteration=lgbm_upper_model.best_iteration)

# Enforce monotonicity of bands post-hoc to avoid tiny inversions
lower_conf_int = np.minimum.reduce([y_pred_lower, y_pred_median, y_pred_upper])
upper_conf_int = np.maximum.reduce([y_pred_lower, y_pred_median, y_pred_upper])
y_median = np.clip(y_pred_median, lower_conf_int, upper_conf_int)

In [12]:
# Create a Bokeh plotting function to plot the actual, ML predicted, and forecast power generation
def ML_future_energy_predict(dataframe, x_axis_column, y_axis_columns, x_axis_label, y_axis_label, title, feature_columns=None,
                             features_ylabel=None, p=None, normalize=False, other_colors=False, color_plot='black',
                             color_features='black', labels=None, features_labels=None, symbols=None, symbols_features=None,
                             test_pred_col=None, lower_conf_col=None, upper_conf_col=None):

    colors={'violet':'#6E36BB','pink':'#D8BAFF','blue':'#2480D0','cyan':'#00E6E6','green':'#1DD14B',
            'yellow':'#FFD700','orange':'#FF6600','dorange':'#DAA520','red':'#DD082C','black':'#000000',
            'grey':'#D0D0D0','dgrey':'#666666'}
    
    if p is None:
        p = figure(title=title, height=900, width=1600, x_axis_type="datetime",
                   tools="reset, hover, zoom_in, zoom_out, box_zoom, wheel_zoom, pan, save")
    
        p.title.text_font_size = '20pt'
        p.yaxis.axis_label = y_axis_label
        p.xaxis.axis_label_text_font_size = "20pt"
        p.xaxis.major_label_text_font_size = "20pt"
        p.xaxis.axis_label_text_font = "times"
        p.xaxis.axis_label_text_color = "black"
        p.xaxis.major_tick_in = 10
        p.xaxis.major_tick_out = 0
        p.xaxis.minor_tick_in = 4
        p.xaxis.minor_tick_out = 0
        p.xaxis.major_tick_line_width = 2
        p.xaxis.axis_label = x_axis_label
        p.yaxis.axis_label_text_font_size = "20pt"
        p.yaxis.major_label_text_font_size = "20pt"
        p.yaxis.axis_label_text_font = "times"
        p.yaxis.axis_label_text_color = "black"
        p.yaxis.major_tick_in = 10
        p.yaxis.major_tick_out = 0
        p.yaxis.minor_tick_in = 4
        p.yaxis.minor_tick_out = 0
        p.yaxis.major_tick_line_width = 2
        p.xaxis.major_label_orientation = 120
        p.xaxis.ticker.desired_num_ticks = 8

        # Format the x-axis datetime labels.
        p.xaxis.formatter = DatetimeTickFormatter(
            minutes="%d-%m-%y %H:%M",
            hours="%d-%m-%y %H:%M",
            days="%d-%m-%y %H:%M",
            months="%d-%m-%y %H:%M",
            years="%d-%m-%y %H:%M"
        )

    if other_colors:
        color_plot_values = color_plot
    else:
        if isinstance(color_plot, list): 
            color_plot_values = [colors[c] for c in color_plot]
        else:
            color_plot_values = colors[color_plot]

    # Create a loop to plot each column data as given by y_axis_columns. Note that these columns will have the same y-range along
    # the primary y-axis.
    color_index = 0
    marker_index = 0
    label_index = 0

    for i, column in enumerate(y_axis_columns):
        label = labels[label_index]

        if normalize:
            max_primary_data = dataframe[y_axis_columns].max().max()
        else:
            max_primary_data = 1

        # Only show the first column by default, hide others
        visible = True if i == 0 else False

        # Scatter plot with symbols.
        if symbols:
            p.scatter(dataframe[x_axis_column], dataframe[column]/max_primary_data, size=10,
                      marker=symbols[marker_index], color=color_plot_values[color_index], alpha=0.5,
                      legend_label=label, visible=visible)
            
        # Add a line to connect the symbols
        p.line(dataframe[x_axis_column], dataframe[column]/max_primary_data,
               line_width=2, color=color_plot_values[color_index], alpha=0.5, legend_label=label,
               visible=visible)

        marker_index = marker_index + 1
        color_index = color_index + 1
        label_index = label_index + 1

    if test_pred_col is not None and lower_conf_col is not None and upper_conf_col is not None:
        mask = ~dataframe[lower_conf_col].isna()
        if normalize:
            max_primary_data = dataframe[y_axis_columns].max().max()
        else:
            max_primary_data = 1
        p.varea(x=dataframe[x_axis_column][mask],
                y1=dataframe[lower_conf_col][mask]/max_primary_data,
                y2=dataframe[upper_conf_col][mask]/max_primary_data,
                fill_color="grey", fill_alpha=0.3,
                legend_label="1sigma Confidence Interval")

    # Create a loop to plot the feature data, if given. Note, secondary y axis must be the same for all features!
    if feature_columns:

        if other_colors:
            color_plot_values = color_features
        else:
            if isinstance(color_features, list): 
                color_plot_values = [colors[c] for c in color_features]
            else:
                color_plot_values = colors[color_features]

        # Specify the secondary y-range for plotting the features data. All features data will be plotted on the same secondary
        # y-axis range, so find minimum and maximum values for the first feature
        y_range = p.y_range
        p.extra_y_ranges['features'] = y_range
        label_index = 0
        color_index = 0
        marker_index = 0

        for i, feature_column in enumerate(feature_columns):

            feature_label = features_labels[label_index]

            if normalize:
                max_features_data = dataframe[feature_column].max()
            else:
                max_features_data = 1

            # Only show the first column by default, hide others
            visible = True if i == 0 else False

            # Scatter plot with symbols. Only plot the energy usage as a function of time.
            if symbols_features:
                p.scatter(dataframe[x_axis_column], dataframe[feature_column]/max_features_data,
                          y_range_name="features", size=10, marker=symbols_features[marker_index],
                          color=color_plot_values[color_index], alpha=0.5, legend_label=feature_label,
                          visible=visible)
            # Add a line to connect the symbols
            p.line(dataframe[x_axis_column], dataframe[feature_column]/max_features_data, y_range_name="features",
                   line_width=2, color=color_plot_values[color_index], alpha=0.5,
                   legend_label=feature_label, visible=visible)
        
            marker_index = marker_index + 1
            color_index = color_index + 1
            label_index = label_index + 1

        # Add the secondary y-axis
        secondary_y_axis = LinearAxis(y_range_name="features", axis_label=" ".join(features_ylabel))
        p.add_layout(secondary_y_axis, 'right')

        secondary_y_axis.axis_label_text_font_size = "20pt"  # Adjust label font size
        secondary_y_axis.major_label_text_font_size = "20pt"  # Adjust tick label font size
        secondary_y_axis.axis_label_text_font = "times"
        secondary_y_axis.axis_label_text_color = "black"
        secondary_y_axis.axis_label_text_font_size = "16pt"
        secondary_y_axis.axis_label_text_font_style = "normal"
        secondary_y_axis.major_tick_in = 10
        secondary_y_axis.major_tick_out = 0
        secondary_y_axis.minor_tick_in = 4
        secondary_y_axis.minor_tick_out = 0

    # Allow user to hide/show plot features.
    if not getattr(p, "_legend_initialized", False):
        from bokeh.models import Legend
        # If there's no legend yet, add a single one
        if len(p.legend) == 0:
            p.add_layout(Legend(), 'right')
        # Configure legend only once
        p.legend.click_policy="hide"
        p.legend.background_fill_alpha = 0.3
        p.legend.border_line_alpha = 0.2
        # mark as initialized so subsequent calls won’t add another
        p._legend_initialized = True

    return(p)

In [13]:
# Let's plot residuals of predictions - actual from both the training and testing sets
# to get a better understand of what is going on, look for trends, etc. Additionally, I will plot a histogram
# of the residuals to see if they are skewed.

from bokeh.plotting import figure, show
from bokeh.layouts import row, column
from bokeh.models import Span, Legend
import numpy as np
import pandas as pd

def plot_train_test_residuals_bokeh(y_train, model_fit, y_test, test_pred, test_index, title="Train/Test Residuals Analysis"):
    # Calculate residuals
    resid_train = y_train - model_fit.fittedvalues
    resid_test = y_test - test_pred

    # If your index isn't datetime, use integer index for plotting
    if not np.issubdtype(y_train.index.dtype, np.datetime64):
        x_train = np.arange(len(y_train))
        x_test = test_index if np.issubdtype(type(test_index[0]), np.datetime64) else np.arange(len(y_test))
        x_axis_type = None
    else:
        x_train = y_train.index
        x_test = test_index
        x_axis_type = "datetime"
    
    # Time Series Residual Plot
    p = figure(title=title, height=800, width=1200, x_axis_type=x_axis_type, tools="pan,wheel_zoom,box_zoom,reset,save")
    l1 = p.line(x_train, resid_train, color="navy", alpha=0.6, legend_label="Train Residuals", line_width=2)
    l2 = p.line(x_test, resid_test, color="firebrick", alpha=0.8, legend_label="Test Residuals", line_width=2)
    p.add_layout(Span(location=0, dimension='width', line_color='black', line_dash='dashed'))
    p.legend.location = "top_left"
    p.legend.click_policy = "hide"
    p.xaxis.axis_label = "Datetime"
    p.yaxis.axis_label = r"$$\mathrm{Residuals\ (Actual - Predicted)}$$"
    # Format the x-axis datetime labels.
    p.xaxis.formatter = DatetimeTickFormatter(
        minutes="%d-%m-%y %H:%M",
        hours="%d-%m-%y %H:%M",
        days="%d-%m-%y %H:%M",
        months="%d-%m-%y %H:%M",
        years="%d-%m-%y %H:%M"
    )
    
    # Residual Histogram
    # Use pandas dropna to avoid plotting nans (can happen if short train/test)
    train_resid_nonan = resid_train.dropna()
    test_resid_nonan = pd.Series(resid_test).dropna()
    hist1, edges1 = np.histogram(train_resid_nonan, bins=50)
    hist2, edges2 = np.histogram(test_resid_nonan, bins=50)
    
    p_hist = figure(title="Residuals Histogram", height=800, width=1200)
    p_hist.quad(top=hist1, bottom=0, left=edges1[:-1], right=edges1[1:], fill_color="navy", line_color="white",
                alpha=0.6, legend_label="Train")
    p_hist.quad(top=hist2, bottom=0, left=edges2[:-1], right=edges2[1:], fill_color="firebrick", line_color="white",
                alpha=0.4, legend_label="Test")
    p_hist.legend.location = "top_left"
    p_hist.legend.click_policy = "hide"
    p_hist.xaxis.axis_label = r"$$\mathrm{Residuals\ (Actual - Predicted)}$$"
    p_hist.yaxis.axis_label = "Count"

    return p, p_hist

In [14]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ----- Helpers -----
def mean_absolute_percentage_error(y_true, y_pred):
    """MAPE (%) with zero-protection."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = (y_true != 0) & np.isfinite(y_true) & np.isfinite(y_pred)
    if not np.any(mask):
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0

def smape(y_true, y_pred):
    """sMAPE (%)"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred)
    mask = (denom != 0) & np.isfinite(denom) & np.isfinite(diff)
    if not np.any(mask):
        return np.nan
    return np.mean(diff[mask] / denom[mask]) * 100.0

def safe_metrics(y_true, y_pred):
    """Compute metrics after masking invalid pairs."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if not np.any(mask):
        return dict(MSE=np.nan, RMSE=np.nan, MAE=np.nan, MAPE=np.nan, sMAPE=np.nan, R2=np.nan)
    yt, yp = y_true[mask], y_pred[mask]
    return {
        "MSE": mean_squared_error(yt, yp),
        "RMSE": np.sqrt(mean_squared_error(yt, yp)),
        "MAE": mean_absolute_error(yt, yp),
        "MAPE": mean_absolute_percentage_error(yt, yp),
        "sMAPE": smape(yt, yp),
        "R2": r2_score(yt, yp)
    }

In [ ]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------

# ----- Expect these to already exist -----
# y_test: Series of actual solar generation on test set
# y_pred_test: ndarray/Series of LightGBM predictions for test set (aligned with y_test)
# forecast_test: Series of Day-Ahead solar forecast (aligned with y_test)
# current_dir: your working dir (used for saving)

# --- Full Test Set Metrics ---
metrics = {}

y_true_full = y_test.values
y_lgbm_full = np.asarray(y_pred_mean)
y_forecast_full = forecast_test.values

metrics['All Test Data'] = {
    'LightGBM': safe_metrics(y_true_full, y_lgbm_full),
    'Forecast': safe_metrics(y_true_full, y_forecast_full)
}

# --- First Day Only (first 24 hours of test set) ---
cut = 24
y_true_day1 = y_test.values[:cut]
y_lgbm_day1 = y_lgbm_full[:cut]
y_forecast_day1 = y_forecast_full[:cut]

metrics['First 24 Hours'] = {
    'LightGBM': safe_metrics(y_true_day1, y_lgbm_day1),
    'Forecast': safe_metrics(y_true_day1, y_forecast_day1)
}

# (Optional) RMSE Skill Score vs Forecast
def rmse_skill(y_true, y_model, y_bench):
    rmse_m = np.sqrt(mean_squared_error(y_true, y_model))
    rmse_b = np.sqrt(mean_squared_error(y_true, y_bench))
    return 1.0 - (rmse_m / rmse_b) if rmse_b > 0 else np.nan

metrics['All Test Data']['Skill_vs_Forecast'] = {
    "RMSE Skill": rmse_skill(y_true_full, y_lgbm_full, y_forecast_full)
}
metrics['First 24 Hours']['Skill_vs_Forecast'] = {
    "RMSE Skill": rmse_skill(y_true_day1, y_lgbm_day1, y_forecast_day1)
}

# ----- Pretty table (MultiIndex columns) -----
df_metrics = pd.DataFrame({
    (section, model): metrics[section][model]
    for section in metrics.keys()
    for model in metrics[section].keys()
})

# Order columns nicely
cols_order = []
for section in ['All Test Data', 'First 24 Hours']:
    for model in ['LightGBM', 'Forecast', 'Skill_vs_Forecast']:
        if model in metrics[section]:
            cols_order.append((section, model))
df_metrics = df_metrics[cols_order]

df_disp = df_metrics.round(3)
print(df_disp)

# ----- Save to CSV / TXT / MD -----
direct_out = current_dir + '/output/results/LightGBM'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out_csv = direct_out + '/lightgbm_vs_forecast_metrics_solar_CAMS.csv'

df_metrics.to_csv(filename_out_csv)

filename_out_txt = direct_out + '/lightgbm_vs_forecast_metrics_solar_CAMS.txt'
with open(filename_out_txt, 'w', encoding='utf-8') as f:
    f.write("LightGBM vs Day-Ahead Forecast — Metrics (Solar Generation with CAMS)\n")
    f.write("=" * 70 + "\n\n")
    f.write(df_disp.to_string())
    f.write("\n")

filename_out_md = direct_out + '/lightgbm_vs_forecast_metrics_solar_CAMS.md'
with open(filename_out_md, 'w', encoding='utf-8') as f:
    f.write("# LightGBM vs Day-Ahead Forecast — Metrics (Solar Generation with CAMS)\n\n")
    md_df = df_disp.copy()
    md_df.columns = [f"{lvl0} – {lvl1}" for (lvl0, lvl1) in md_df.columns]
    f.write(md_df.to_markdown(index=True))
    f.write("\n")

print(f"\nSaved metrics:\n- CSV: {filename_out_csv}\n- TXT: {filename_out_txt}\n- MD:  {filename_out_md}")

In [72]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------

# --- 6) Feature importance
# Importance by gain and by split (counts)
feat_names = mean_model.feature_name()
imp_gain   = mean_model.feature_importance(importance_type='gain')
imp_split  = mean_model.feature_importance(importance_type='split')

fi_df = (pd.DataFrame({
            'feature': feat_names,
            'gain': imp_gain,
            'split': imp_split
        })
        .assign(gain_pct=lambda d: 100 * d['gain'] / d['gain'].sum() if d['gain'].sum() > 0 else 0)
        .sort_values('gain', ascending=False)
        .reset_index(drop=True))

print("\nTop 20 features by gain:\n", fi_df.head(20))

filename_out = direct_out + '/lgbm_feature_importance_gain_split_Solar_CAMS.csv'
fi_df.to_csv(filename_out, index=False)

# fi = (pd.Series(mean_model.feature_importances_, index=X_train.columns)
#         .sort_values(ascending=False))
# print("\nTop 20 features:\n", fi.head(20))


Top 20 features by gain:
                            feature          gain  split   gain_pct
0                         sol_elev  2.208321e+11    152  52.586408
1                     GHI_weighted  8.281954e+10    149  19.721691
2               daylight_length_hr  2.138202e+10    935   5.091667
3                              GHI  1.240881e+10     33   2.954892
4                      day_of_year  1.148827e+10    536   2.735684
5                  day_of_year_sin  6.863568e+09    564   1.634411
6                  day_of_year_cos  5.906828e+09    406   1.406584
7            utc_time_day_fraction  5.288378e+09     57   1.259313
8            date_fraction_of_year  4.186608e+09    136   0.996950
9                     day_hour_sin  2.551665e+09     13   0.607624
10             GHI_weighted_lag24h  2.546469e+09     41   0.606387
11                    day_hour_cos  2.520980e+09     48   0.600317
12         Cloud_coverage_weighted  2.489896e+09     80   0.592915
13                           month 

In [ ]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------

# --- 7) Build plot_df for your Bokeh function
N = len(train) + len(test)
full_pred = np.full(N, np.nan)
full_pred[len(train):] = y_pred_mean

full_pred_med = np.full(N, np.nan)
full_pred_med[len(train):] = y_median

test_rec = test.copy()
test_rec['LightGBM Prediction Mean'] = y_pred_mean
test_rec['LightGBM Prediction Median'] = y_median
test_rec['Forecast Solar Day Ahead'] = data.loc[split_idx:, forecast_col].to_numpy(copy=True)

train_pred = np.full(N, np.nan)
train_pred[:split_idx] = mean_model.predict(X_train, num_iteration=mean_model.best_iteration)

plot_df_lgbm = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Solar': data[target_col].to_numpy(copy=True),
    'LightGBM Prediction Mean': full_pred,
    'LightGBM Prediction Median': full_pred_med,
    'LightGBM Train Prediction': train_pred,
    'Forecast Solar Day Ahead': data[forecast_col].to_numpy(copy=True)
})

plot_df_lgbm['LGBM Lower Bound'] = np.nan
plot_df_lgbm['LGBM Upper Bound'] = np.nan
plot_df_lgbm.loc[split_idx:, 'LGBM Lower Bound'] = lower_conf_int
plot_df_lgbm.loc[split_idx:, 'LGBM Upper Bound'] = upper_conf_int

# Add significant cols
for col in feature_cols_model:
    plot_df_lgbm[col] = data[col].to_numpy(copy=True)

p = ML_future_energy_predict(plot_df_lgbm, 'utc_time', ['Actual Solar', 'LightGBM Prediction Mean', 'LightGBM Prediction Median',
                                                        'LightGBM Train Prediction', 'Forecast Solar Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Solar\ Power\ Generation\ (MWh)}$$",
                             'Actual Solar Power, LightGBM Solar Power Prediction'
                             + ' Forecast Solar Day Ahead vs UTC Time, Solar Elevation, and GHI Weighted',
                             feature_columns=['sol_elev', 'GHI_weighted'],
                             features_ylabel=[r"$$\mathrm{Solar\ elevation\ (^{\circ})\ \&\ GHI\ weighted\ (W\ m^{-2})}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'pink', 'black', 'blue'],
                             color_features=['green', 'cyan'],
                             labels=['Actual Solar', 'LightGBM Predicted Mean Solar Power', 'LightGBM Predicted Median Solar Power',
                                     'LightGBM Train Prediction', 'Forecast Solar Day Ahead'],
                             features_labels=['Solar Elevation', 'Solar Flux'],
                             symbols=['star', 'triangle', 'inverted_triangle', 'square', 'diamond'],
                             symbols_features=['circle', 'cross'], test_pred_col='LightGBM Train Prediction',
                             lower_conf_col='LGBM Lower Bound', upper_conf_col='LGBM Upper Bound')

show(p)

# Save plots.
direct_out = current_dir + '/output/results/LightGBM/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_Solar_Power_LightGBM_CAMS.html'

title = 'Actual Solar Power, LightGBM Solar Power Prediction, Forecast Solar Day Ahead vs UTC Time, Solar Elevation, and GHI Weighted'

save(p, filename_out, title=title)

In [74]:
class FittedAdapter:
    def __init__(self, fitted_series):
        self.fittedvalues = fitted_series

y_pred_train = mean_model.predict(X_train, num_iteration=mean_model.best_iteration)
lgbm_train = FittedAdapter(pd.Series(y_pred_train, index=y_train.index))

p, p_hist = plot_train_test_residuals_bokeh(y_train, lgbm_train, y_test, test_rec['LightGBM Prediction Mean'], test_rec.index,
                                            title="Train/Test Residuals Analysis for Solar Power Generation with CAMS")

# Save plots.
direct_out = current_dir + '/output/results/LightGBM/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Residuals_actual_predicted_train_test_CAMS.html'

title = 'Train/Test Residuals Analysis for Solar Power Generation with CAMS'

save(p, filename_out, title=title)

filename_out = direct_out + '/Histogram_residuals_train_test_CAMS.html'

title = 'Histogram of the Residuals for Solar Power Generation with CAMS'

save(p_hist, filename_out, title=title)

/var/folders/wv/ww_f6bg15tv52kq7ghvh0w880000gp/T/ipykernel_99947/589584555.py:22: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(p, filename_out, title=title)
/var/folders/wv/ww_f6bg15tv52kq7ghvh0w880000gp/T/ipykernel_99947/589584555.py:28: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(p_hist, filename_out, title=title)


'/Users/u8010412/Library/CloudStorage/Dropbox/Data_Science/Projects/Kaggle_energy_data/archive/output/results/LightGBM/plots/Histogram_residuals_train_test_CAMS.html'

In [15]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import mean_squared_error

# ---------- helpers ----------
def _lgb_callbacks(early_stopping_rounds=200, log_period=100, enable_log=True, evals_result=None):
    cbs = [lgb.early_stopping(early_stopping_rounds)]
    if evals_result is not None:
        cbs.append(lgb.record_evaluation(evals_result))
    if enable_log and log_period:
        cbs.append(lgb.log_evaluation(log_period))
    return cbs

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def _build_30day_walkforward_splits(X_train, initial_train_frac=0.4, val_span_days=30):
    """
    Build walk-forward splits with:
      - fixed-length training window (first `initial_train_frac` of data)
      - validation window of `val_span_days` days
      - step size = `val_span_days` days
    Returns: list of (train_idx, valid_idx) arrays (integer positions).
    """
    # Assumes X_train index is DatetimeIndex (UTC hourly)
    idx = X_train.index.sort_values()
    n = len(idx)
    if n == 0:
        raise ValueError("X_train is empty or has no index.")

    start = idx[0]
    end = idx[-1]

    # Initial training window length (in hours)
    init_end_time = idx[int(n * initial_train_frac)]
    train_len_hours = int((init_end_time - start).total_seconds() // 3600.0)
    if train_len_hours <= 0:
        raise ValueError("Initial training window too short. Increase initial_train_frac.")

    # Fixed validation span and step (both 30 days by default)
    val_span = pd.Timedelta(days=val_span_days)
    step = pd.Timedelta(days=val_span_days)

    # First validation window starts immediately after the initial training window
    first_val_start = init_end_time + pd.Timedelta(hours=1)  # next hour after training end
    splits = []

    val_start = first_val_start
    while val_start <= end:
        val_end = min(val_start + val_span - pd.Timedelta(hours=1), end)  # inclusive end hour

        # Slide training window so it ends just before validation starts
        tr_end_time = val_start - pd.Timedelta(hours=1)
        tr_start_time = tr_end_time - pd.Timedelta(hours=train_len_hours)

        # Clip to available range
        tr_start_time = max(tr_start_time, start)
        tr_end_time = min(tr_end_time, end)

        # Collect indices
        tr_idx = X_train.loc[tr_start_time:tr_end_time].index
        va_idx = X_train.loc[val_start:val_end].index

        if len(tr_idx) > 0 and len(va_idx) > 0:
            splits.append((
                X_train.index.get_indexer(tr_idx),
                X_train.index.get_indexer(va_idx)
            ))

        # Advance by 30 days
        val_start = val_start + step

    if len(splits) == 0:
        raise ValueError("No 30-day splits were generated. Check your index / initial_train_frac.")
        
    return splits


def backtest_score_lgbm_logged(X, y, splits, params, cat_cols=None, early_stopping_rounds=200, log_period=0,
                               trial_num=None, log_file_path=None):

    """
    Walk-forward score over 'splits' (list of (train_idx, valid_idx) tuples).
    Returns the average metric (lower is better).
    Also logs per-window train/valid RMSE to `log_file_path` if provided.
    """
    
    scores = []
    train_losses = []
    val_losses = []

    # Open log in append mode once per trial
    log_f = open(log_file_path, 'a', encoding='utf-8') if log_file_path else None
    if log_f and trial_num is not None:
        log_f.write(f"\n==== Trial {trial_num} ====\n")
        log_f.write("Params:\n")
        for k, v in params.items():
            log_f.write(f"  {k}: {v}\n")

    for k, (tr_idx, va_idx) in enumerate(splits, start=1):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

        dtr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols or None, free_raw_data=True)
        dva = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols or None, free_raw_data=True, reference=dtr)

        evals_result = {}
        booster = lgb.train(
            params=params,
            train_set=dtr,
            num_boost_round=5000,
            valid_sets=[dtr, dva],
            valid_names=['train', 'valid'],
            callbacks=_lgb_callbacks(
                early_stopping_rounds=early_stopping_rounds,
                log_period=log_period,
                enable_log=bool(log_period),
                evals_result=evals_result
            )
        )

        best_it = booster.best_iteration
        # Get last recorded train/valid metrics at best_it
        tr_hist = evals_result.get('train', {}).get('rmse', [])
        va_hist = evals_result.get('valid', {}).get('rmse', [])
        tr_rmse = tr_hist[best_it-1] if best_it and len(tr_hist) >= best_it else (tr_hist[-1] if tr_hist else np.nan)
        va_rmse = va_hist[best_it-1] if best_it and len(va_hist) >= best_it else (va_hist[-1] if va_hist else np.nan)

        y_hat = booster.predict(X_va, num_iteration=best_it)
        fold_rmse = rmse(y_va, y_hat)

        scores.append(fold_rmse)
        train_losses.append(tr_rmse)
        val_losses.append(va_rmse)

        if log_f:
            log_f.write(f"  Window {k:02d}: best_iter={best_it}, train_RMSE={tr_rmse:.4f}, valid_RMSE={va_rmse:.4f}\n")

    avg_valid = float(np.mean(scores))
    avg_train = float(np.nanmean(train_losses)) if len(train_losses) else np.nan

    if log_f:
        log_f.write(f"--> Trial mean valid RMSE: {avg_valid:.4f}, mean train RMSE: {avg_train:.4f}\n")
        log_f.flush()
        log_f.close()

    return avg_valid

def tune_lgbm_with_optuna_logged(X_cv, y_cv, splits,
                                  cat_cols,
                                  n_trials=60,
                                  metric='rmse',
                                  random_seed=42,
                                  log_dir=None):

    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, "optuna_trials_log.txt")

    def objective(trial: optuna.Trial):
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'verbosity': -1,
            'boosting_type': 'gbdt',
            'seed': random_seed,
            'n_jobs': -1,

            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 16, 512, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 16),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 1000, log=True),

            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 0, 10),

            'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        }

        score = backtest_score_lgbm_logged(
            X=X_cv, y=y_cv, splits=splits, params=params, cat_cols=cat_cols,
            early_stopping_rounds=200, log_period=0,
            trial_num=trial.number, log_file_path=log_file
        )
        return score

    # Create a sampler with a fixed seed
    sampler = TPESampler(seed=random_seed)

    study = optuna.create_study(direction='minimize', study_name='lgbm_mean_ts_30d_cv70_init40', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    # Append final best to the log
    with open(os.path.join(log_dir, "optuna_trials_log.txt"), 'a', encoding='utf-8') as f:
        f.write("\n==== BEST TRIAL ====\n")
        f.write(f"Trial #{study.best_trial.number}\n")
        f.write(f"Value (mean valid RMSE): {study.best_value:.6f}\n")
        f.write("Best params:\n")
        for k, v in study.best_trial.params.items():
            f.write(f"  {k}: {v}\n")

    best_params = study.best_trial.params
    best_params.update({
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': random_seed,
        'n_jobs': -1
    })
    return best_params, study

In [ ]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------


# ---------- run tuning with monthly Walk Forward Validation ----------
# Assumes you already have X_train, y_train (indexed by utc_time), and cat_cols
N_total = len(data)
cv_end = int(N_total * 0.70)  # first 70% for CV, rest for testing
X_cv = data.iloc[:cv_end][feature_cols_model].copy()
y_cv = data.iloc[:cv_end][target_col].copy()

# Ensure hourly DateTimeIndex (you already do this elsewhere, just re-assert)
cv_idx = pd.DatetimeIndex(data.iloc[:cv_end]['utc_time'].values, name='utc_time')

X_cv.index = cv_idx
y_cv.index = cv_idx

X_cv  = X_cv.asfreq('h')
y_cv  = y_cv.asfreq('h')

splits_30d = _build_30day_walkforward_splits(X_cv, initial_train_frac=0.4, val_span_days=30)
print(f"Generated {len(splits_30d)} WFV splits on first 70% of data.")

direct_out = current_dir + '/output/results/LightGBM/'

log_dir = os.path.join(direct_out, "tuning_logs_CAMS")

print('log_dir: ', log_dir)

best_params, study = tune_lgbm_with_optuna_logged(
    X_cv=X_cv, y_cv=y_cv,
    splits=splits_30d,
    cat_cols=cat_cols,
    n_trials=60,
    metric='rmse',
    random_seed=42,
    log_dir=log_dir
)
print("Best mean valid RMSE:", study.best_value)
print("Best params:", best_params)

In [16]:
# 6) Quantile models (P16 / P50 / P84) reusing tuned params & same 30-day val tail
def train_quantile_with_val(alpha: float, best_rounds: int):
    """
    Train a quantile LGBM model with early stopping on the same 30-day validation tail.
    Uses best_rounds (from mean model) as an upper cap on boosting.
    """
    q_params = best_params.copy()
    q_params.update({
        'objective': 'quantile',
        'alpha': alpha,
        'metric': 'quantile',
        'verbosity': -1
    })
    model_q = lgb.train(
        params=q_params,
        train_set=fit_set_final,
        num_boost_round=int(best_rounds),
        valid_sets=[fit_set_final, val_set_final],
        valid_names=["train","val"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=max(50, best_rounds // 10)),
            lgb.log_evaluation(max(50, best_rounds // 10))
        ]
    )
    return model_q

In [ ]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------

# Base plotting DataFrame over CV slice
plot_cv_df = pd.DataFrame({
    'utc_time': X_cv.index,
    'Actual Solar': y_cv.values
})

# Start the plot with the actual series
p_wfv = ML_future_energy_predict(
    dataframe=plot_cv_df.copy(),
    x_axis_column='utc_time',
    y_axis_columns=['Actual Solar'],
    x_axis_label=r"$$\mathrm{UTC\ (DD\!-\!MM\!-\!YY\ \ HH:MM)}$$",
    y_axis_label=r"$$\mathrm{Solar\ Power\ Generation\ (MWh)}$$",
    title='Walk-Forward Validation Windows — Actual vs Best-Param Predictions',
    p=None,
    other_colors=False,
    color_plot=['dorange'],
    labels=['Actual Solar'],
    symbols=['star']
)

# Colors to cycle through for windows
win_colors = ['red', 'blue', 'green', 'purple', 'black', 'cyan', 'pink', 'orange', 'navy', 'olive']

for w, (tr_idx, va_idx) in enumerate(splits_30d, start=1):
    X_tr_w, y_tr_w = X_cv.iloc[tr_idx], y_cv.iloc[tr_idx]
    X_va_w, y_va_w = X_cv.iloc[va_idx], y_cv.iloc[va_idx]

    dtr_w = lgb.Dataset(X_tr_w, label=y_tr_w, categorical_feature=cat_cols or None, free_raw_data=True)
    dva_w = lgb.Dataset(X_va_w, label=y_va_w, categorical_feature=cat_cols or None, free_raw_data=True, reference=dtr_w)

    # short training with early stopping; you can increase num_boost_round
    evals_result_w = {}
    booster_w = lgb.train(
        params=best_params,
        train_set=dtr_w,
        num_boost_round=5000,
        valid_sets=[dtr_w, dva_w],
        valid_names=['train','valid'],
        callbacks=_lgb_callbacks(early_stopping_rounds=200, log_period=0, evals_result=evals_result_w)
    )
    pred_va = booster_w.predict(X_va_w, num_iteration=booster_w.best_iteration)

    # Add a column with NaNs outside the window so only that window draws
    col_name = f'Pred Window {w:02d}'

    X_va_index = pd.DatetimeIndex(X_va_w.index).tz_localize(None) if hasattr(X_va_w.index, 'tz') \
                 and X_va_w.index.tz is not None else X_va_w.index
    pred_series = pd.Series(pred_va, index=X_va_index, name=col_name)
    
    plot_seg = plot_cv_df.copy()
    plot_seg['utc_time'] = pd.DatetimeIndex(plot_seg['utc_time']).tz_localize(None)

    plot_seg = plot_seg.merge(pred_series, how='left', left_on='utc_time', right_index=True)
    # plot_seg[col_name] = np.nan
    # plot_seg.loc[X_va_w.index, col_name] = pred_va

    # Layer onto the same Bokeh fig
    p_wfv = ML_future_energy_predict(
        dataframe=plot_seg,
        x_axis_column='utc_time',
        y_axis_columns=[col_name],
        x_axis_label=r"$$\mathrm{UTC\ (DD\!-\!MM\!-\!YY\ \ HH:MM)}$$",
        y_axis_label=r"$$\mathrm{Solar\ Power\ (MWh)}$$",
        title='Walk-Forward Validation Windows — Actual vs Best-Param Predictions',
        p=p_wfv,
        other_colors=True,
        color_plot=[win_colors[(w-1) % len(win_colors)]],
        labels=[col_name],
        symbols=['circle']
    )

    # # ensure only one legend exists on the figure you are reusing
    # for lg in list(p_wfv.legend):
    #     p_wfv.remove_layout(lg)
    
    # p_wfv.add_layout(Legend(), 'right')
    # p_wfv.legend.click_policy = "hide"

show(p_wfv)

# Save plots.
direct_out = current_dir + '/output/results/LightGBM/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_Solar_Power_LightGBM_CAMS_step_ahead_CV_windows_best_params.html'

title = 'Actual Solar Power and LightGBM Solar Power Prediction Windows vs UTC Time'

save(p_wfv, filename_out, title=title)

# 1) Make sure the train/test indices are sorted & hourly (you already call .asfreq('h'))
X_tr_all = X_train.sort_index()
y_tr_all = y_train.loc[X_tr_all.index]

# 2) Build 30-day validation tail (720 hours); fallback if train shorter than 30 days
val_end = X_tr_all.index.max()
val_start = val_end - pd.Timedelta(days=30) + pd.Timedelta(hours=1)

if val_start < X_tr_all.index.min():
    # fallback by position (last 720 hours if available)
    tail_len = min(720, len(X_tr_all))
    X_val_final = X_tr_all.iloc[-tail_len:]
    y_val_final = y_tr_all.iloc[-tail_len:]
    X_tr_final  = X_tr_all.iloc[:-tail_len]
    y_tr_final  = y_tr_all.iloc[:-tail_len]
else:
    val_mask    = (X_tr_all.index >= val_start) & (X_tr_all.index <= val_end)
    X_val_final = X_tr_all.loc[val_mask]
    y_val_final = y_tr_all.loc[val_mask]
    X_tr_final  = X_tr_all.loc[~val_mask]
    y_tr_final  = y_tr_all.loc[~val_mask]

print("Train final range:", X_tr_final.index.min(), "→", X_tr_final.index.max(), f"({len(X_tr_final)} rows)")
print("Valid final range:", X_val_final.index.min(), "→", X_val_final.index.max(), f"({len(X_val_final)} rows)")

# 3) LightGBM datasets
fit_set_final = lgb.Dataset(X_tr_final, label=y_tr_final, categorical_feature=cat_cols or None, free_raw_data=False)
val_set_final = lgb.Dataset(X_val_final, label=y_val_final, categorical_feature=cat_cols or None, free_raw_data=False)

# 4) Train MEAN model with early stopping on the 30-day tail
mean_model_final = lgb.train(
    params=best_params,                # from Optuna (objective='regression', metric='rmse', etc.)
    train_set=fit_set_final,
    num_boost_round=5000,
    valid_sets=[fit_set_final, val_set_final],
    valid_names=["train","val"],
    callbacks=[lgb.early_stopping(stopping_rounds=200), lgb.log_evaluation(200)]
)
best_iter_mean = mean_model_final.best_iteration
print("Mean model best_iteration:", best_iter_mean)

# 5) Predictions (TRAIN fitted and TEST forecast)
y_fitted_mean = pd.Series(mean_model_final.predict(X_train, num_iteration=best_iter_mean), index=X_train.index,
                          name='lgbm_mean_fitted')
y_pred_mean = pd.Series(mean_model_final.predict(X_test, num_iteration=best_iter_mean), index=X_test.index,
                        name='lgbm_mean_pred')

# Train quantile models
q16_model = train_quantile_with_val(alpha=0.16, best_rounds=best_iter_mean)
q50_model = train_quantile_with_val(alpha=0.50, best_rounds=best_iter_mean)  # "median"
q84_model = train_quantile_with_val(alpha=0.84, best_rounds=best_iter_mean)

# Predict on TRAIN (fitted-style) and TEST for each quantile, if you’d like
pred_p16_train = pd.Series(q16_model.predict(X_train, num_iteration=q16_model.best_iteration), index=X_train.index)
pred_p50_train = pd.Series(q50_model.predict(X_train, num_iteration=q50_model.best_iteration), index=X_train.index)
pred_p84_train = pd.Series(q84_model.predict(X_train, num_iteration=q84_model.best_iteration), index=X_train.index)

pred_p16_test  = pd.Series(q16_model.predict(X_test,  num_iteration=q16_model.best_iteration), index=X_test.index,
                           name='lgbm_p16_pred')
pred_p50_test  = pd.Series(q50_model.predict(X_test,  num_iteration=q50_model.best_iteration), index=X_test.index,
                           name='lgbm_p50_pred')
pred_p84_test  = pd.Series(q84_model.predict(X_test,  num_iteration=q84_model.best_iteration), index=X_test.index,
                           name='lgbm_p84_pred')

# Clean up bands (avoid inversions)
lower_band_test = np.minimum.reduce([pred_p16_test.values, pred_p50_test.values, pred_p84_test.values])
upper_band_test = np.maximum.reduce([pred_p16_test.values, pred_p50_test.values, pred_p84_test.values])
pred_median_test = np.clip(pred_p50_test.values, lower_band_test, upper_band_test)

pred_p16_test = pd.Series(lower_band_test, index=X_test.index, name='lgbm_lower_16')
pred_p84_test = pd.Series(upper_band_test, index=X_test.index, name='lgbm_upper_84')
pred_median_test = pd.Series(pred_median_test, index=X_test.index, name='lgbm_median_pred')

final_refit_outputs = {
    'mean_model_final': mean_model_final,
    'best_iter_mean': best_iter_mean,
    'y_fitted_mean': y_fitted_mean,
    'y_pred_mean': y_pred_mean,
    'pred_p16_test': pred_p16_test,
    'pred_p50_test': pred_median_test,   # median
    'pred_p84_test': pred_p84_test
}

# Assemble into a convenient DataFrame for plotting or metrics
quant_pred_df = pd.DataFrame({
    'utc_time': X_test.index,
    'LGBM_P16': pred_p16_test.values,
    'LGBM_P50': pred_median_test.values,
    'LGBM_P84': pred_p84_test.values,
    'LGBM_Mean': y_pred_mean.values  # from your mean_model_final earlier
}).set_index('utc_time')

# (Optional) get feature importance from the mean model safely
# LightGBM native Booster uses feature_importance/feature_name (not feature_importances_)
fi = pd.Series(
    mean_model_final.feature_importance(importance_type='gain'),
    index=mean_model_final.feature_name()
).sort_values(ascending=False)

print("\nTop 20 features (mean model, gain):\n", fi.head(20))

direct_out = current_dir + '/output/results/LightGBM/tuning_logs/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

mean_model_final.save_model(direct_out + 'lgbm_mean_CAMS.txt')
q16_model.save_model(direct_out + 'lgbm_q16_CAMS.txt')
q50_model.save_model(direct_out + 'lgbm_q50_CAMS.txt')
q84_model.save_model(direct_out + 'lgbm_q84_CAMS.txt')

In [ ]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------
# ---------- 1) Metrics using your existing helpers ----------
# Full test
y_true_full      = y_test.values
y_lgbm_full_mean = y_pred_mean.values
y_lgbm_full_med  = np.asarray(pred_median_test)
y_fc_full        = forecast_test.values

h24 = min(24, len(y_test))
y_true_24        = y_test.values[:h24]
y_lgbm_24_mean   = y_pred_mean.values[:h24]
y_lgbm_24_med    = np.asarray(pred_median_test)[:h24]
y_fc_24          = forecast_test.values[:h24]

# --- Metrics dict (adds LightGBM median) ---
metrics = {
    "All Test Data": {
        "LightGBM mean": safe_metrics(y_true_full, y_lgbm_full_mean) | {
            "RMSE Skill (vs Forecast)": rmse_skill(y_true_full, y_lgbm_full_mean, y_fc_full)
        },
        "LightGBM median": safe_metrics(y_true_full, y_lgbm_full_med) | {
            "RMSE Skill (vs Forecast)": rmse_skill(y_true_full, y_lgbm_full_med, y_fc_full)
        },
        "Forecast": safe_metrics(y_true_full, y_fc_full) | {
            "RMSE Skill (vs Forecast)": rmse_skill(y_true_full, y_fc_full, y_fc_full)
        },
    },
    "First 24 Hours": {
        "LightGBM mean": safe_metrics(y_true_24, y_lgbm_24_mean) | {
            "RMSE Skill (vs Forecast)": rmse_skill(y_true_24, y_lgbm_24_mean, y_fc_24)
        },
        "LightGBM median": safe_metrics(y_true_24, y_lgbm_24_med) | {
            "RMSE Skill (vs Forecast)": rmse_skill(y_true_24, y_lgbm_24_med, y_fc_24)
        },
        "Forecast": safe_metrics(y_true_24, y_fc_24) | {
            "RMSE Skill (vs Forecast)": rmse_skill(y_true_24, y_fc_24, y_fc_24)
        },
    }
}

# --- Build MultiIndex dataframe ---
df_metrics = pd.DataFrame({
    (section, model): metrics[section][model]
    for section in metrics
    for model in metrics[section]
})

# Desired column order (list of 2-tuples!)
order = [
    ('All Test Data','LightGBM mean'),
    ('All Test Data','LightGBM median'),
    ('All Test Data','Forecast'),
    ('First 24 Hours','LightGBM mean'),
    ('First 24 Hours','LightGBM median'),
    ('First 24 Hours','Forecast'),
]
df_metrics = df_metrics.reindex(columns=pd.MultiIndex.from_tuples(order))

print(df_metrics.round(3))

# ---------- 2) Save report (CSV + pretty TXT) ----------
direct_out = current_dir + '/output/results/LightGBM'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out_csv = direct_out + '/lightgbm_vs_forecast_metrics_solar_CAMS_step_ahead_CV.csv'

df_metrics.to_csv(filename_out_csv)

filename_out_txt = direct_out + '/lightgbm_vs_forecast_metrics_solar_CAMS_step_ahead_CV.txt'
txt_df = df_metrics.copy()
txt_df.columns = [f"{a} – {b}" for a,b in txt_df.columns]
with open(filename_out_txt, 'w', encoding='utf-8') as f:
    f.write("LightGBM vs Day-Ahead Forecast — Metrics (Solar Generation)\n")
    f.write("=" * 72 + "\n\n")
    f.write(txt_df.round(3).to_string())
    f.write("\n")

filename_out_md = direct_out + '/lightgbm_vs_forecast_metrics_solar_CAMS_step_ahead_CV.md'
md_df = df_metrics.copy()
md_df.columns = [f"{a} – {b}" for a,b in md_df.columns]
with open(filename_out_md, 'w', encoding='utf-8') as f:
    f.write("# LightGBM vs Day-Ahead Forecast — Metrics (Solar Generation)\n\n")
    f.write(md_df.round(3).to_markdown(index=True))
    f.write("\n")

# --- 3) Feature importance
# Importance by gain and by split (counts)
feat_names = mean_model_final.feature_name()
imp_gain   = mean_model_final.feature_importance(importance_type='gain')
imp_split  = mean_model_final.feature_importance(importance_type='split')

fi_df = (pd.DataFrame({
            'feature': feat_names,
            'gain': imp_gain,
            'split': imp_split
        })
        .assign(gain_pct=lambda d: 100 * d['gain'] / d['gain'].sum() if d['gain'].sum() > 0 else 0)
        .sort_values('gain', ascending=False)
        .reset_index(drop=True))

print("\nTop 20 features by gain:\n", fi_df.head(20))

filename_out = direct_out + '/lgbm_CAMS_feature_importance_gain_split_step_ahead_CV.csv'
fi_df.to_csv(filename_out, index=False)

# ---------- 4) Build plotting dataframe for Bokeh (Actual vs Preds + 1σ P16–P84 band) ----------
full_pred_mean = np.full(N, np.nan); full_pred_mean[split_idx:] = y_pred_mean.values
full_pred_med = np.full(N, np.nan); full_pred_med[split_idx:] = pred_median_test
full_fit  = np.full(N, np.nan); full_fit[:split_idx]  = y_fitted_mean
full_p16  = np.full(N, np.nan); full_p16[split_idx:]  = pred_p16_test
full_p84  = np.full(N, np.nan); full_p84[split_idx:]  = pred_p84_test

plot_df_lgbm_CV = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Solar': data[target_col].to_numpy(copy=True),
    'LightGBM Prediction Mean': full_pred_mean,               # test preds mean
    'LightGBM Prediction Median': full_pred_med,          # test preds median
    'LightGBM Train Prediction': full_fit,              # train fitted
    'Forecast Solar Day Ahead': data[forecast_col].to_numpy(copy=True),
    'LightGBM Lower Bound': full_p16,               # P16
    'LightGBM Upper Bound': full_p84                # P84
})

for col in feature_cols_model:
    plot_df_lgbm_CV[col] = data[col].to_numpy(copy=True)

# ---------- 5) Main plot (your existing function; just pass CI columns) ----------
p = ML_future_energy_predict(plot_df_lgbm_CV, 'utc_time', ['Actual Solar', 'LightGBM Prediction Mean', 'LightGBM Prediction Median',
                                                        'LightGBM Train Prediction', 'Forecast Solar Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Solar\ Power\ Generation\ (MWh)}$$",
                             'Actual Solar Power, LightGBM Solar Power Prediction'
                             + ' Forecast Solar Day Ahead vs UTC Time, Solar Elevation, and GHI Weighted for Step Ahead CV',
                             feature_columns=['sol_elev', 'GHI_weighted'],
                             features_ylabel=[r"$$\mathrm{Solar\ elevation\ (^{\circ})\ \&\ GHI\ weighted\ (W\ m^{-2})}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'pink', 'black', 'blue'],
                             color_features=['green', 'cyan'],
                             labels=['Actual Solar', 'LightGBM Predicted Mean Solar Power', 'LightGBM Predicted Median Solar Power',
                                     'LightGBM Train Prediction', 'Forecast Solar Day Ahead'],
                             features_labels=['Solar Elevation', 'Weighted GHI'],
                             symbols=['star', 'triangle', 'inverted_triangle', 'square', 'diamond'],
                             symbols_features=['circle', 'cross'], test_pred_col='LightGBM Prediction Mean',
                             lower_conf_col='LightGBM Lower Bound', upper_conf_col='LightGBM Upper Bound')

show(p)

# Save plots.
direct_out = current_dir + '/output/results/LightGBM/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_Solar_Power_LightGBM_CAMS_step_ahead_CV.html'

title = 'Actual Solar Power, LightGBM Solar Power Prediction, Forecast Solar Day Ahead vs UTC Time'

save(p, filename_out, title=title)

# ---------- 6) Residual plots (reuse your earlier residual plotting if you have it) ----------
# If you still have the function `plot_train_test_residuals_bokeh` that expects a model with `.fittedvalues`,
# instead just pass arrays using the array-based version. If you only have the model-based version,
# quickly create a lightweight shim so it has `.fittedvalues`:
y_pred_train = y_train
lgbm_train = FittedAdapter(pd.Series(y_fitted_mean, index=y_train.index))

p, p_hist = plot_train_test_residuals_bokeh(y_train, lgbm_train, y_test, y_pred_mean, X_test.index,
                                            title="Train/Test Residuals Analysis for Solar Power Generation LightGBM step ahead CV")

# Save plots.
direct_out = current_dir + '/output/results/LightGBM/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Residuals_actual_predicted_train_test_CAMS_step_ahead_CV.html'

title = 'Train/Test Residuals Analysis for Solar Power Generation'

save(p, filename_out, title=title)

filename_out = direct_out + '/Histogram_residuals_train_test_CAMS_step_ahead_CV.html'

title = 'Histogram of the Residuals for Solar Power Generation'

save(p_hist, filename_out, title=title)

In [ ]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------
from tqdm.auto import tqdm
# --- CONFIG: choose training mode for each daily refit ---
USE_VAL_TAIL = True     # True = Mode A (adaptive early stopping); False = Mode B (fixed rounds on all data)
VAL_TAIL_DAYS = 30
EARLY_STOP_ROUNDS = 200
LOG_PERIOD = 200
STEP_HOURS = 24

# best_params: tuned params for MEAN model (objective='regression', metric='rmse', etc.)
# best_iter_mean_cv: the best_iteration you kept from CV (only needed for Mode B)

def _callbacks(early_stopping_rounds=EARLY_STOP_ROUNDS, log_period=LOG_PERIOD):
    cbs = [lgb.early_stopping(early_stopping_rounds)]
    if log_period and log_period > 0:
        cbs.append(lgb.log_evaluation(log_period))
    return cbs

df_all = data.copy(deep=True)
df_all = (df_all
          .set_index('utc_time')
          .sort_index()
          .asfreq('h'))  # ensure hourly, fills missing hours with NaN rows

# Ensure categorical dtypes for the full frame
for c in (cat_cols or []):
    if c in df_all.columns:
        df_all[c] = df_all[c].astype('category')

# Split out full spans
X_all = df_all[feature_cols_model].copy()
y_all = df_all[target_col].copy()
forecast_all = df_all[forecast_col].copy()
idx_all = X_all.index  # the master index we’ll use for containers

# If you still need train/test boundaries for rolling 24h:
N_total = len(df_all)
split_idx_all = int(N_total * 0.70)
test_start_time = idx_all[split_idx_all]
test_end_time   = idx_all[-1]

# (Optional) For plotting containers restricted to the test span only:
test_mask = (idx_all >= test_start_time) & (idx_all <= test_end_time)

# Containers for full-span predictions
y_pred_mean_full = pd.Series(np.nan, index=idx_all)
y_pred_p16_full  = pd.Series(np.nan, index=idx_all)
y_pred_p50_full  = pd.Series(np.nan, index=idx_all)
y_pred_p84_full  = pd.Series(np.nan, index=idx_all)

# current_start = test_start_time
step = pd.Timedelta(days=1)  # 24h step
window_starts = pd.date_range(start=test_start_time, end=test_end_time, freq=step)

# while current_start <= test_end_time:
for current_start in tqdm(window_starts, desc="Rolling 24h predictions"):
    pred_start = current_start
    pred_end = min(pred_start + step - pd.Timedelta(hours=1), test_end_time)
    # pred_start = current_start
    # pred_end   = min(pred_start + pd.Timedelta(hours=STEP_HOURS) - pd.Timedelta(hours=1), test_end_time)
    pred_index = X_all.loc[pred_start:pred_end].index
    if len(pred_index) == 0:
        break

    train_end = pred_start - pd.Timedelta(hours=1)
    tr_index = X_all.loc[:train_end].index
    if len(tr_index) == 0:
        break

    if USE_VAL_TAIL:
        # ------------------ Mode A: adaptive early stopping with val tail ------------------
        val_tail_hours = 24 * VAL_TAIL_DAYS
        tail_start = train_end - pd.Timedelta(hours=val_tail_hours) + pd.Timedelta(hours=1)
        tail_start = max(tail_start, tr_index.min())

        X_tr_core = X_all.loc[:tail_start - pd.Timedelta(hours=1)].copy()
        y_tr_core = y_all.loc[X_tr_core.index].copy()
        X_val_tail = X_all.loc[tail_start:train_end].copy()
        y_val_tail = y_all.loc[X_val_tail.index].copy()

        # fallback if core empty
        if len(X_tr_core) == 0:
            if len(X_val_tail) >= 48:
                X_tr_core = X_all.loc[:train_end].iloc[:-48].copy()
                y_tr_core = y_all.loc[X_tr_core.index].copy()
                X_val_tail = X_all.loc[:train_end].iloc[-48:].copy()
                y_val_tail = y_all.loc[X_val_tail.index].copy()
            else:
                split_pos = max(1, int(len(tr_index) * 0.8))
                X_tr_core = X_all.loc[tr_index[:split_pos]].copy()
                y_tr_core = y_all.loc[X_tr_core.index].copy()
                X_val_tail = X_all.loc[tr_index[split_pos:]].copy()
                y_val_tail = y_all.loc[X_val_tail.index].copy()

        # # reassert categoricals
        # for c in (cat_cols or []):
        #     if c in X_tr_core.columns:
        #         X_tr_core[c] = X_tr_core[c].astype('category')
        #         X_val_tail[c] = X_val_tail[c].astype('category')

        dtr = lgb.Dataset(X_tr_core, label=y_tr_core, categorical_feature=cat_cols or None, free_raw_data=True)
        dva = lgb.Dataset(X_val_tail, label=y_val_tail, categorical_feature=cat_cols or None, free_raw_data=True, reference=dtr)

        # Mean model
        booster = lgb.train(
            params=best_params,
            train_set=dtr,
            num_boost_round=5000,
            valid_sets=[dtr, dva],
            valid_names=['train','valid'],
            callbacks=_callbacks()
        )
        best_it_mean = booster.best_iteration

        # Quantile helper
        def _train_quantile(alpha):
            qp = best_params.copy()
            qp.update({'objective': 'quantile', 'alpha': alpha, 'metric': 'quantile', 'verbosity': -1})
            return lgb.train(
                params=qp,
                train_set=dtr,
                num_boost_round=int(best_it_mean),
                valid_sets=[dtr, dva],
                valid_names=['train','val'],
                callbacks=[lgb.early_stopping(max(50, best_it_mean // 10)),
                           lgb.log_evaluation(max(50, best_it_mean // 10))]
            )

        q16 = _train_quantile(0.16)
        q50 = _train_quantile(0.50)
        q84 = _train_quantile(0.84)

        # Predict next 24h
        X_pred = X_all.loc[pred_index].copy()
        # for c in (cat_cols or []):
        #     if c in X_pred.columns:
        #         X_pred[c] = X_pred[c].astype('category')

        y_hat_mean = booster.predict(X_pred, num_iteration=best_it_mean)
        y_hat_p16  = q16.predict(X_pred, num_iteration=q16.best_iteration)
        y_hat_p50  = q50.predict(X_pred, num_iteration=q50.best_iteration)
        y_hat_p84  = q84.predict(X_pred, num_iteration=q84.best_iteration)

    else:
        # ------------------ Mode B: full training (no val), fixed num_boost_round from CV ------------------
        # use all data up to train_end
        X_tr_core = X_all.loc[:train_end].copy()
        y_tr_core = y_all.loc[X_tr_core.index].copy()
        # for c in (cat_cols or []):
        #     if c in X_tr_core.columns:
        #         X_tr_core[c] = X_tr_core[c].astype('category')

        dtr = lgb.Dataset(X_tr_core, label=y_tr_core, categorical_feature=cat_cols or None, free_raw_data=True)

        # Mean
        booster = lgb.train(
            params=best_params,
            train_set=dtr,
            num_boost_round=int(best_iter_mean_cv),  # fixed from CV
            valid_sets=[dtr], valid_names=['train'], callbacks=[lgb.log_evaluation(LOG_PERIOD)]
        )

        # Quantile helper
        def _train_quantile(alpha):
            qp = best_params.copy()
            qp.update({'objective': 'quantile', 'alpha': alpha, 'metric': 'quantile', 'verbosity': -1})
            return lgb.train(
                params=qp,
                train_set=dtr,
                num_boost_round=int(best_iter_mean_cv),
                valid_sets=[dtr], valid_names=['train'], callbacks=[lgb.log_evaluation(LOG_PERIOD)]
            )

        q16 = _train_quantile(0.16)
        q50 = _train_quantile(0.50)
        q84 = _train_quantile(0.84)

        X_pred = X_all.loc[pred_index].copy()
        # for c in (cat_cols or []):
        #     if c in X_pred.columns:
        #         X_pred[c] = X_pred[c].astype('category')

        y_hat_mean = booster.predict(X_pred, num_iteration=booster.num_trees())
        y_hat_p16  = q16.predict(X_pred,     num_iteration=q16.num_trees())
        y_hat_p50  = q50.predict(X_pred,     num_iteration=q50.num_trees())
        y_hat_p84  = q84.predict(X_pred,     num_iteration=q84.num_trees())

    # Enforce monotone bands and store
    lb = np.minimum.reduce([y_hat_p16, y_hat_p50, y_hat_p84])
    ub = np.maximum.reduce([y_hat_p16, y_hat_p50, y_hat_p84])
    med = np.clip(y_hat_p50, lb, ub)

    y_pred_mean_full.loc[pred_index] = y_hat_mean
    y_pred_p16_full.loc[pred_index]  = lb
    y_pred_p50_full.loc[pred_index]  = med
    y_pred_p84_full.loc[pred_index]  = ub

    # current_start = pred_end + pd.Timedelta(hours=1)

In [ ]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------

# ---------- Build plotting DataFrame with bands ----------
plot_roll_quant = pd.DataFrame({
    'utc_time': X_all.index,
    'Actual Solar': y_all.values,
    'LightGBM 24h Prediction Mean': y_pred_mean_full.values,
    'LightGBM 24h Prediction Median': y_pred_p50_full.values,
    'LightGBM Lower Band': y_pred_p16_full.values,
    'LightGBM Upper Band': y_pred_p84_full.values,
    'Forecast Solar Day Ahead': forecast_all.values
})

# if 'GHI_weighted' in data.columns:
#     plot_roll_quant['GHI_weighted'] = data['GHI_weighted'].values

for col in feature_cols_model:
    plot_roll_quant[col] = data[col].values

# ---------- 5) Main plot (your existing function; just pass CI columns) ----------
# Draw the single clean plot over whole test span (your function supports CI via test_pred_col/lower/upper)
p = ML_future_energy_predict(
    dataframe=plot_roll_quant,
    x_axis_column='utc_time', 
    y_axis_columns=['Actual Solar', 'LightGBM 24h Prediction Mean', 'LightGBM 24h Prediction Median', 'Forecast Solar Day Ahead'],
    x_axis_label=r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$",
    y_axis_label=r"$$\mathrm{Solar\ Power\ Generation\ (MWh)}$$",
    title='Actual vs LightGBM 24h Rolling Prediction (Mean and Median) with 1 sigma confidence bounds vs Day-Ahead Forecast',
    feature_columns=(['sol_elev', 'GHI_weighted'] if 'sol_elev' in plot_roll_quant.columns and 'GHI_weighted'
                     in plot_roll_quant.columns else None),
    features_ylabel=[r"$$\mathrm{Solar\ elevation\ (^{\circ})\ \&\ GHI\ weighted\ (W\ m^{-2})}$$"] \
                     if 'sol_elev' in plot_roll_quant.columns and 'GHI_weighted' in plot_roll_quant.columns else None,
    p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'pink', 'blue'],
    color_features=['green', 'cyan'],
    labels=['Actual Solar', 'LightGBM 24h Prediction (Mean)', 'LightGBM 24h Prediction Median', 'Forecast Solar Day Ahead'],
    features_labels=(['Solar Elevation', 'Weighted GHI'] if 'sol_elev' in plot_roll_quant.columns and 'GHI_weighted'
                     in plot_roll_quant.columns else None),
    symbols=['star', 'triangle', 'inverted_triangle', 'diamond'],
    symbols_features=(['circle', 'cross'] if 'sol_elev' in plot_roll_quant.columns and 'GHI_weighted'
                     in plot_roll_quant.columns else None),
    test_pred_col='LightGBM 24h Prediction Mean',
    lower_conf_col='LightGBM Lower Band',
    upper_conf_col='LightGBM Upper Band')

show(p)

# Save plots.
direct_out = current_dir + '/output/results/LightGBM/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_Solar_Power_CAMS_LightGBM_24hr_step_ahead_CV_windows.html'

title = 'Actual vs LightGBM 24h Rolling Prediction (Mean and Median) with 1 sigma confidence bounds vs Day-Ahead Forecast'

save(p, filename_out, title=title)

In [93]:
# ------------------------------- Model 1 -- No solar generation roll or lag features ----------------------------
# ========= Overall metrics on the full TEST span (LightGBM 24h mean/median vs Forecast) =========
# Assumes you already have: plot_roll_quant, test_start_time, test_end_time,
# and your helpers: safe_metrics, rmse_skill.

# 1) Slice the test range
mask_test = (plot_roll_quant['utc_time'] >= test_start_time) & (plot_roll_quant['utc_time'] <= test_end_time)

yt = plot_roll_quant.loc[mask_test, 'Actual Solar'].values
yp_mean = plot_roll_quant.loc[mask_test, 'LightGBM 24h Prediction Mean'].values
yp_median = plot_roll_quant.loc[mask_test, 'LightGBM 24h Prediction Median'].values
yf = plot_roll_quant.loc[mask_test, 'Forecast Solar Day Ahead'].values

# 2) Build metrics dict (LightGBM Mean + Median + Forecast) incl. RMSE Skill vs Forecast
metrics_overall = {
    "All Test Data": {
        "LightGBM 24h Mean": safe_metrics(yt, yp_mean) | {
            "RMSE Skill (vs Forecast)": rmse_skill(yt, yp_mean, yf)
        },
        "LightGBM 24h Median": safe_metrics(yt, yp_median) | {
            "RMSE Skill (vs Forecast)": rmse_skill(yt, yp_median, yf)
        },
        "Forecast": safe_metrics(yt, yf) | {
            "RMSE Skill (vs Forecast)": rmse_skill(yt, yf, yf)
        },
    }
}

# 3) Pretty DataFrame (MultiIndex columns)
df_overall = pd.DataFrame({
    (section, model): metrics_overall[section][model]
    for section in metrics_overall
    for model in metrics_overall[section]
})

# Optional: column order
df_overall = df_overall[[('All Test Data','LightGBM 24h Mean'),
                         ('All Test Data','LightGBM 24h Median'),
                         ('All Test Data','Forecast')]]

print("\n=== Overall Test Metrics (LightGBM 24h Mean/Median vs Forecast) ===")
print(df_overall.round(3))

# 4) Save CSV / TXT / Markdown
direct_out = current_dir + '/output/results/LightGBM'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/lightgbm_rolling24h_vs_forecast_metrics_overall_with_median_CAMS.csv'

df_overall.to_csv(filename_out)

filename_out = direct_out + '/lightgbm_rolling24h_vs_forecast_metrics_overall_with_median_CAMS.txt'
with open(filename_out, 'w', encoding='utf-8') as f:
    f.write("LightGBM 24h Rolling Prediction (Mean & Median) vs Day-Ahead Forecast — Overall Test Metrics\n")
    f.write("=" * 94 + "\n\n")
    f.write(df_overall.round(3).to_string())
    f.write("\n")

filename_out = direct_out + '/lightgbm_rolling24h_vs_forecast_metrics_overall_with_median_CAMS.md'
with open(filename_out, 'w', encoding='utf-8') as f:
    f.write("# LightGBM 24h Rolling Prediction (Mean & Median) vs Day-Ahead Forecast — Overall Test Metrics\n\n")
    md_df = df_overall.copy()
    md_df.columns = [f"{lvl0} – {lvl1}" for (lvl0, lvl1) in md_df.columns]
    f.write(md_df.round(3).to_markdown(index=True))
    f.write("\n")

#print(f"\nSaved overall metrics (mean & median) to:\n- {csv_path}\n- {txt_path}\n- {md_path}")


=== Overall Test Metrics (LightGBM 24h Mean/Median vs Forecast) ===
                             All Test Data                               
                         LightGBM 24h Mean LightGBM 24h Median   Forecast
MSE                             314928.280          408988.606  46701.181
RMSE                               561.185             639.522    216.105
MAE                                327.498             325.197    130.143
MAPE                               248.903             203.864     54.601
sMAPE                               60.211              50.966     36.964
R2                                   0.883               0.848      0.983
RMSE Skill (vs Forecast)            -1.597              -1.959      0.000


In [94]:
import dill
import inspect

def is_picklable(obj) -> bool:
    try:
        dill.dumps(obj)
        return True
    except Exception:
        return False

def is_bokeh_object(obj) -> bool:
    return obj.__class__.__module__.startswith("bokeh.")

skip_names = {"In", "Out", "exit", "quit", "get_ipython", "dill"}

bad_names = []
session_vars = {}

for name, val in list(globals().items()):
    if name in skip_names or name.startswith("_"):
        continue
    if inspect.ismodule(val) or inspect.isfunction(val):
        continue
    if is_bokeh_object(val):
        bad_names.append((name, "bokeh object"))
        continue
    if is_picklable(val):
        session_vars[name] = val
    else:
        bad_names.append((name, type(val).__name__))

print(bad_names)

[('Label', 'bokeh object'), ('ColumnDataSource', 'bokeh object'), ('CustomJS', 'bokeh object'), ('Slider', 'bokeh object'), ('Whisker', 'bokeh object'), ('BoxAnnotation', 'bokeh object'), ('Arrow', 'bokeh object'), ('OpenHead', 'bokeh object'), ('Span', 'bokeh object'), ('figure', 'bokeh object'), ('Legend', 'bokeh object'), ('LinearAxis', 'bokeh object'), ('Range1d', 'bokeh object'), ('LabelSet', 'bokeh object'), ('HoverTool', 'bokeh object'), ('DatetimeTickFormatter', 'bokeh object'), ('fit_set', 'Dataset'), ('val_set', 'Dataset'), ('p', 'bokeh object'), ('p_hist', 'bokeh object'), ('p_wfv', 'bokeh object'), ('dtr_w', 'Dataset'), ('dva_w', 'Dataset'), ('fit_set_final', 'Dataset'), ('val_set_final', 'Dataset'), ('dtr', 'Dataset'), ('dva', 'Dataset')]


In [95]:
# Now try dumping the session
with open("notebook_session.pkl", "wb") as f:
    dill.dump(session_vars, f)

In [17]:
import dill

# Restore the notebook session
with open("notebook_session.pkl", "rb") as f:
    session_vars = dill.load(f)

globals().update(session_vars)

In [20]:
# ----------------------------- Model 2 -- Solar generation using roll and lag features --------------------------

# Now I will create a LightGBM model using the solar generation lag and roll features.
# The solar roll and lag features will need to be handled recursively when making model predictions using the predicted solar
# generation power to avoid data leakage.

# feature_cols_w_solar_roll_lag = features_cols + target_related_col

# data_lag_roll_solar = national_solar_df[[target_col, forecast_col, datetime_col] + feature_cols_w_solar_roll_lag].copy()

# data_lag_roll_solar = data_lag_roll_solar.sort_values(datetime_col).reset_index(drop=True)
# data_lag_roll_solar = data_lag_roll_solar.set_index(datetime_col).asfreq('h').reset_index()
# data_lag_roll_solar = data_lag_roll_solar.dropna(subset=[target_col] + feature_cols_w_solar_roll_lag).copy()

# target_related_col = ['generation solar_roll3h', 'generation solar_roll6h', 'generation solar_lag1h', 'generation solar_lag2h',
#                       'generation solar_lag3h', 'generation solar_lag24h']


def _callbacks(early_stopping_rounds=EARLY_STOP_ROUNDS, log_period=LOG_PERIOD):
    cbs = [lgb.early_stopping(early_stopping_rounds)]
    if log_period and log_period > 0:
        cbs.append(lgb.log_evaluation(log_period))
    return cbs

def _best_iter_or_num_trees(model):
    bi = getattr(model, 'best_iteration', None)
    if bi is None or bi <= 0:
        return model.num_trees()
    return bi

def _safe_hist_value(y_hist, ts, fallback_value):
    if ts in y_hist.index and pd.notna(y_hist.loc[ts]):
        return float(y_hist.loc[ts])
    return float(fallback_value)

def _rolling_from_history(y_hist, ts, window_h, fallback_value):
    """
    Computes shift(1).rolling(window=window_h, min_periods=1).mean() at timestamp ts,
    using only the previous `window_h` hours from y_hist.
    """
    vals = []
    for h in range(1, window_h + 1):
        t_prev = ts - pd.Timedelta(hours=h)
        vals.append(_safe_hist_value(y_hist, t_prev, fallback_value))
    return float(np.mean(vals))

def _build_target_related_feature_values(ts, y_hist, target_related_col, fallback_value):
    """
    Build a dict of recursive target-related feature values for one timestamp.
    """
    feature_values = {}

    if 'generation solar_lag1h' in target_related_col:
        feature_values['generation solar_lag1h'] = _safe_hist_value(
            y_hist, ts - pd.Timedelta(hours=1), fallback_value
        )

    if 'generation solar_lag2h' in target_related_col:
        feature_values['generation solar_lag2h'] = _safe_hist_value(
            y_hist, ts - pd.Timedelta(hours=2), fallback_value
        )

    if 'generation solar_lag3h' in target_related_col:
        feature_values['generation solar_lag3h'] = _safe_hist_value(
            y_hist, ts - pd.Timedelta(hours=3), fallback_value
        )

    if 'generation solar_lag24h' in target_related_col:
        feature_values['generation solar_lag24h'] = _safe_hist_value(
            y_hist, ts - pd.Timedelta(hours=24), fallback_value
        )

    if 'generation solar_roll3h' in target_related_col:
        feature_values['generation solar_roll3h'] = _rolling_from_history(
            y_hist, ts, window_h=3, fallback_value=fallback_value
        )

    if 'generation solar_roll6h' in target_related_col:
        feature_values['generation solar_roll6h'] = _rolling_from_history(
            y_hist, ts, window_h=6, fallback_value=fallback_value
        )

    return feature_values

# def _update_target_related_features_for_row(X_row, ts, y_hist, target_related_col, fallback_value):
#     """
#     Overwrite the target-related lag/roll features in a single-row DataFrame
#     using only historical actuals + previously generated predictions.
#     """
#     if 'generation solar_lag1h' in target_related_col:
#         X_row.loc[ts, 'generation solar_lag1h'] = _safe_hist_value(
#             y_hist, ts - pd.Timedelta(hours=1), fallback_value
#         )

#     if 'generation solar_lag2h' in target_related_col:
#         X_row.loc[ts, 'generation solar_lag2h'] = _safe_hist_value(
#             y_hist, ts - pd.Timedelta(hours=2), fallback_value
#         )

#     if 'generation solar_lag3h' in target_related_col:
#         X_row.loc[ts, 'generation solar_lag3h'] = _safe_hist_value(
#             y_hist, ts - pd.Timedelta(hours=3), fallback_value
#         )

#     if 'generation solar_lag24h' in target_related_col:
#         X_row.loc[ts, 'generation solar_lag24h'] = _safe_hist_value(
#             y_hist, ts - pd.Timedelta(hours=24), fallback_value
#         )

#     if 'generation solar_roll3h' in target_related_col:
#         X_row.loc[ts, 'generation solar_roll3h'] = _rolling_from_history(
#             y_hist, ts, window_h=3, fallback_value=fallback_value
#         )

#     if 'generation solar_roll6h' in target_related_col:
#         X_row.loc[ts, 'generation solar_roll6h'] = _rolling_from_history(
#             y_hist, ts, window_h=6, fallback_value=fallback_value
#         )

#     return X_row

def recursive_predict_block(
    X_all,
    y_hist_initial,
    pred_index,
    target_related_col,
    booster,
    q16,
    q50,
    q84,
    num_it_mean,
    num_it_q16,
    num_it_q50,
    num_it_q84,
    feature_cols_model,
    clip_nonnegative=True,
    recursive_source='mean'
):
    """
    Sequentially predict one forecast block, updating target lag/roll features
    from previous actuals/predictions so there is no leakage.

    Returns:
    - mean predictions
    - p16 predictions
    - p50 predictions
    - p84 predictions
    - DataFrame of recursive target-related features used at each predicted timestamp
    """
    y_hist = y_hist_initial.copy()
    first_valid_hist = y_hist.dropna()
    if len(first_valid_hist) == 0:
        raise ValueError("y_hist_initial has no valid values.")
    fallback_value = float(first_valid_hist.iloc[0])

    y_hat_mean_block = []
    y_hat_p16_block = []
    y_hat_p50_block = []
    y_hat_p84_block = []

    recursive_feature_rows = []

    for ts in pred_index:
        X_row = X_all.loc[[ts], feature_cols_model].copy()

        # Build and store recursive target-related features
        feature_values = _build_target_related_feature_values(
            ts=ts,
            y_hist=y_hist,
            target_related_col=target_related_col,
            fallback_value=fallback_value
        )

        for col, val in feature_values.items():
            X_row.loc[ts, col] = val

        # Save the recursive features used for this timestamp
        recursive_feature_rows.append(
            {'utc_time': ts, **feature_values}
        )

        # Predict this single timestamp
        pred_mean = float(booster.predict(X_row, num_iteration=num_it_mean)[0])
        pred_p16  = float(q16.predict(X_row, num_iteration=num_it_q16)[0])
        pred_p50  = float(q50.predict(X_row, num_iteration=num_it_q50)[0])
        pred_p84  = float(q84.predict(X_row, num_iteration=num_it_q84)[0])

        if clip_nonnegative:
            pred_mean = max(pred_mean, 0.0)
            pred_p16  = max(pred_p16, 0.0)
            pred_p50  = max(pred_p50, 0.0)
            pred_p84  = max(pred_p84, 0.0)

        y_hat_mean_block.append(pred_mean)
        y_hat_p16_block.append(pred_p16)
        y_hat_p50_block.append(pred_p50)
        y_hat_p84_block.append(pred_p84)

        # Update recursive history for future hours within the block
        if recursive_source == 'p50':
            y_hist.loc[ts] = pred_p50
        else:
            y_hist.loc[ts] = pred_mean

    recursive_features_block = pd.DataFrame(recursive_feature_rows).set_index('utc_time')

    return (
        np.array(y_hat_mean_block),
        np.array(y_hat_p16_block),
        np.array(y_hat_p50_block),
        np.array(y_hat_p84_block),
        recursive_features_block
    )

# def recursive_predict_block(
#     X_all,
#     y_hist_initial,
#     pred_index,
#     target_related_col,
#     booster,
#     q16,
#     q50,
#     q84,
#     num_it_mean,
#     num_it_q16,
#     num_it_q50,
#     num_it_q84,
#     feature_cols_model,
#     clip_nonnegative=True,
#     recursive_source='mean'
# ):
#     """
#     Sequentially predict one forecast block, updating target lag/roll features
#     from previous actuals/predictions so there is no leakage.
#     """
#     y_hist = y_hist_initial.copy()
#     first_valid_hist = y_hist.dropna()
#     if len(first_valid_hist) == 0:
#         raise ValueError("y_hist_initial has no valid values.")
#     fallback_value = float(first_valid_hist.iloc[0])

#     y_hat_mean_block = []
#     y_hat_p16_block = []
#     y_hat_p50_block = []
#     y_hat_p84_block = []

#     for ts in pred_index:
#         # Start from the exogenous/base row for this timestamp
#         X_row = X_all.loc[[ts], feature_cols_model].copy()

#         # Overwrite the target-derived features recursively
#         X_row = _update_target_related_features_for_row(
#             X_row=X_row,
#             ts=ts,
#             y_hist=y_hist,
#             target_related_col=target_related_col,
#             fallback_value=fallback_value
#         )

#         # Predict this single timestamp
#         pred_mean = float(booster.predict(X_row, num_iteration=num_it_mean)[0])
#         pred_p16  = float(q16.predict(X_row, num_iteration=num_it_q16)[0])
#         pred_p50  = float(q50.predict(X_row, num_iteration=num_it_q50)[0])
#         pred_p84  = float(q84.predict(X_row, num_iteration=num_it_q84)[0])

#         if clip_nonnegative:
#             pred_mean = max(pred_mean, 0.0)
#             pred_p16  = max(pred_p16, 0.0)
#             pred_p50  = max(pred_p50, 0.0)
#             pred_p84  = max(pred_p84, 0.0)

#         y_hat_mean_block.append(pred_mean)
#         y_hat_p16_block.append(pred_p16)
#         y_hat_p50_block.append(pred_p50)
#         y_hat_p84_block.append(pred_p84)

#         # Update recursive history for future hours within the block
#         if recursive_source == 'p50':
#             y_hist.loc[ts] = pred_p50
#         else:
#             y_hist.loc[ts] = pred_mean

#     return (
#         np.array(y_hat_mean_block),
#         np.array(y_hat_p16_block),
#         np.array(y_hat_p50_block),
#         np.array(y_hat_p84_block),
#     )

In [21]:
# ----------------------------- Model 2 -- Solar generation using roll and lag features --------------------------

from tqdm.auto import tqdm

# =========================
# CONFIG
# =========================
USE_VAL_TAIL = True      # True = adaptive early stopping using a 30-day validation tail
VAL_TAIL_DAYS = 30
EARLY_STOP_ROUNDS = 200
LOG_PERIOD = 200
STEP_HOURS = 24          # predict in 24h blocks
TRAIN_FRAC = 0.70        # train/test split for rolling backtest
CLIP_NONNEGATIVE = True  # solar generation should not go negative
RECURSIVE_SOURCE = 'p50'  # 'mean' or 'p50' for updating recursive lag/roll state

# =========================
# Prepare full frame
# =========================
required_cols = [datetime_col, target_col, forecast_col] + feature_cols_w_solar_roll_lag
missing_cols = [c for c in required_cols if c not in data_lag_roll_solar.columns]
if missing_cols:
    raise KeyError(f"These required columns are missing from data_lag_roll_solar: {missing_cols}")

df_all = data_lag_roll_solar[required_cols].copy()
df_all[datetime_col] = pd.to_datetime(df_all[datetime_col])
df_all = df_all.sort_values(datetime_col).set_index(datetime_col)

# Optional sanity check for hourly continuity
expected_idx = pd.date_range(df_all.index.min(), df_all.index.max(), freq='h')
if not expected_idx.equals(df_all.index):
    print("Warning: index is not perfectly continuous hourly. "
          "This code will still run, but recursive lags assume hourly spacing.")

cat_cols = [c for c in ['month', 'day_of_week', 'season'] if c in feature_cols_w_solar_roll_lag]

# Ensure categorical dtypes
for c in cat_cols:
    df_all[c] = df_all[c].astype('category')

# Split out full spans
X_all = df_all[feature_cols_w_solar_roll_lag].copy()
y_all = df_all[target_col].copy()
forecast_all = df_all[forecast_col].copy()
idx_all = X_all.index

# =========================
# Train / test boundary for rolling 24h backtest
# =========================
N_total = len(df_all)
split_idx_all = int(N_total * TRAIN_FRAC)
test_start_time = idx_all[split_idx_all]
test_end_time = idx_all[-1]

# Containers for predictions on full span
y_pred_mean_full = pd.Series(np.nan, index=idx_all, name='pred_mean_model2')
y_pred_p16_full  = pd.Series(np.nan, index=idx_all, name='pred_p16_model2')
y_pred_p50_full  = pd.Series(np.nan, index=idx_all, name='pred_p50_model2')
y_pred_p84_full  = pd.Series(np.nan, index=idx_all, name='pred_p84_model2')

recursive_target_features_full = pd.DataFrame(
    index=idx_all,
    columns=target_related_col,
    dtype=float
)

step = pd.Timedelta(hours=STEP_HOURS)
window_starts = pd.date_range(start=test_start_time, end=test_end_time, freq=step)

# =========================
# Rolling 24h predictions
# =========================
for current_start in tqdm(window_starts, desc="Rolling 24h predictions (Model 2, recursive)"):
    pred_start = current_start
    pred_end = min(pred_start + step - pd.Timedelta(hours=1), test_end_time)

    pred_index = X_all.loc[pred_start:pred_end].index
    if len(pred_index) == 0:
        continue

    train_end = pred_start - pd.Timedelta(hours=1)
    tr_index = X_all.loc[:train_end].index
    if len(tr_index) == 0:
        break

    if USE_VAL_TAIL:
        # ---------- Mode A: daily refit with 30-day validation tail ----------
        val_tail_hours = 24 * VAL_TAIL_DAYS
        tail_start = train_end - pd.Timedelta(hours=val_tail_hours) + pd.Timedelta(hours=1)
        tail_start = max(tail_start, tr_index.min())

        X_tr_core = X_all.loc[:tail_start - pd.Timedelta(hours=1)].copy()
        y_tr_core = y_all.loc[X_tr_core.index].copy()
        X_val_tail = X_all.loc[tail_start:train_end].copy()
        y_val_tail = y_all.loc[X_val_tail.index].copy()

        # fallback if core empty or too small
        if len(X_tr_core) == 0:
            if len(X_val_tail) >= 48:
                X_tr_core = X_all.loc[:train_end].iloc[:-48].copy()
                y_tr_core = y_all.loc[X_tr_core.index].copy()
                X_val_tail = X_all.loc[:train_end].iloc[-48:].copy()
                y_val_tail = y_all.loc[X_val_tail.index].copy()
            else:
                split_pos = max(1, int(len(tr_index) * 0.8))
                X_tr_core = X_all.loc[tr_index[:split_pos]].copy()
                y_tr_core = y_all.loc[X_tr_core.index].copy()
                X_val_tail = X_all.loc[tr_index[split_pos:]].copy()
                y_val_tail = y_all.loc[X_val_tail.index].copy()

        dtr = lgb.Dataset(
            X_tr_core,
            label=y_tr_core,
            categorical_feature=cat_cols or None,
            free_raw_data=True
        )
        dva = lgb.Dataset(
            X_val_tail,
            label=y_val_tail,
            categorical_feature=cat_cols or None,
            free_raw_data=True,
            reference=dtr
        )

        # Mean model
        booster = lgb.train(
            params=best_params,
            train_set=dtr,
            num_boost_round=5000,
            valid_sets=[dtr, dva],
            valid_names=['train', 'valid'],
            callbacks=_callbacks()
        )
        best_it_mean = _best_iter_or_num_trees(booster)

        # Quantile models
        def _train_quantile(alpha):
            qp = best_params.copy()
            qp.update({
                'objective': 'quantile',
                'alpha': alpha,
                'metric': 'quantile',
                'verbosity': -1
            })
            return lgb.train(
                params=qp,
                train_set=dtr,
                num_boost_round=int(best_it_mean),
                valid_sets=[dtr, dva],
                valid_names=['train', 'valid'],
                callbacks=[
                    lgb.early_stopping(max(50, best_it_mean // 10)),
                    lgb.log_evaluation(max(50, best_it_mean // 10))
                ]
            )

        q16 = _train_quantile(0.16)
        q50 = _train_quantile(0.50)
        q84 = _train_quantile(0.84)

        pred_num_it_mean = best_it_mean
        pred_num_it_q16 = _best_iter_or_num_trees(q16)
        pred_num_it_q50 = _best_iter_or_num_trees(q50)
        pred_num_it_q84 = _best_iter_or_num_trees(q84)

    else:
        # ---------- Mode B: no validation tail, fixed rounds from CV ----------
        X_tr_core = X_all.loc[:train_end].copy()
        y_tr_core = y_all.loc[X_tr_core.index].copy()

        dtr = lgb.Dataset(
            X_tr_core,
            label=y_tr_core,
            categorical_feature=cat_cols or None,
            free_raw_data=True
        )

        booster = lgb.train(
            params=best_params,
            train_set=dtr,
            num_boost_round=int(best_iter_mean_cv),
            valid_sets=[dtr],
            valid_names=['train'],
            callbacks=[lgb.log_evaluation(LOG_PERIOD)]
        )

        def _train_quantile(alpha):
            qp = best_params.copy()
            qp.update({
                'objective': 'quantile',
                'alpha': alpha,
                'metric': 'quantile',
                'verbosity': -1
            })
            return lgb.train(
                params=qp,
                train_set=dtr,
                num_boost_round=int(best_iter_mean_cv),
                valid_sets=[dtr],
                valid_names=['train'],
                callbacks=[lgb.log_evaluation(LOG_PERIOD)]
            )

        q16 = _train_quantile(0.16)
        q50 = _train_quantile(0.50)
        q84 = _train_quantile(0.84)

        pred_num_it_mean = booster.num_trees()
        pred_num_it_q16 = q16.num_trees()
        pred_num_it_q50 = q50.num_trees()
        pred_num_it_q84 = q84.num_trees()

    # ---------- Recursive 24h prediction block ----------
    # Only actual history up to train_end is allowed here
    y_hist_initial = y_all.loc[:train_end].copy()

    y_hat_mean, y_hat_p16, y_hat_p50, y_hat_p84, recursive_features_block = recursive_predict_block(
        X_all=X_all,
        y_hist_initial=y_hist_initial,
        pred_index=pred_index,
        target_related_col=target_related_col,
        booster=booster,
        q16=q16,
        q50=q50,
        q84=q84,
        num_it_mean=pred_num_it_mean,
        num_it_q16=pred_num_it_q16,
        num_it_q50=pred_num_it_q50,
        num_it_q84=pred_num_it_q84,
        feature_cols_model=feature_cols_w_solar_roll_lag,
        clip_nonnegative=CLIP_NONNEGATIVE,
        recursive_source=RECURSIVE_SOURCE
    )

    # Store recursive target-related features used for this block
    recursive_target_features_full.loc[pred_index, target_related_col] = (
        recursive_features_block.reindex(pred_index)[target_related_col].values
    )

    # Enforce monotone bands
    lb = np.minimum.reduce([y_hat_p16, y_hat_p50, y_hat_p84])
    ub = np.maximum.reduce([y_hat_p16, y_hat_p50, y_hat_p84])
    med = np.clip(y_hat_p50, lb, ub)

    # Store predictions
    y_pred_mean_full.loc[pred_index] = y_hat_mean
    y_pred_p16_full.loc[pred_index]  = lb
    y_pred_p50_full.loc[pred_index]  = med
    y_pred_p84_full.loc[pred_index]  = ub

# =========================
# Final test-span outputs
# =========================
test_mask = (idx_all >= test_start_time) & (idx_all <= test_end_time)

y_true_test_model2 = y_all.loc[test_mask].copy()
forecast_test_model2 = forecast_all.loc[test_mask].copy()

y_pred_mean_test_model2 = y_pred_mean_full.loc[test_mask].copy()
y_pred_p16_test_model2  = y_pred_p16_full.loc[test_mask].copy()
y_pred_p50_test_model2  = y_pred_p50_full.loc[test_mask].copy()
y_pred_p84_test_model2  = y_pred_p84_full.loc[test_mask].copy()

recursive_target_features_test = recursive_target_features_full.loc[test_mask].copy()

results_model2 = pd.DataFrame({
    'utc_time': y_true_test_model2.index,
    'y_true': y_true_test_model2.values,
    'forecast_solar_day_ahead': forecast_test_model2.values,
    'pred_mean_model2': y_pred_mean_test_model2.values,
    'pred_p16_model2': y_pred_p16_test_model2.values,
    'pred_p50_model2': y_pred_p50_test_model2.values,
    'pred_p84_model2': y_pred_p84_test_model2.values,
}).set_index('utc_time')

# Merge in the recursively generated target features
results_model2 = results_model2.join(recursive_target_features_test)

print("Model 2 recursive forecasting complete.")
print(f"Test span: {test_start_time} to {test_end_time}")
print(f"Results shape: {results_model2.shape}")
display(results_model2.head())

Rolling 24h predictions (Model 2, recursive):   0%|          | 0/439 [00:00<?, ?it/s]

Training until validation scores don't improve for 200 rounds
[200]	train's rmse: 199.801	valid's rmse: 212.852
[400]	train's rmse: 142.796	valid's rmse: 162.546
[600]	train's rmse: 130.844	valid's rmse: 151.338
[800]	train's rmse: 122.696	valid's rmse: 144.871
[1000]	train's rmse: 116.185	valid's rmse: 140.228
[1200]	train's rmse: 110.696	valid's rmse: 136.799
[1400]	train's rmse: 106.141	valid's rmse: 134.171
[1600]	train's rmse: 102.163	valid's rmse: 132.446
[1800]	train's rmse: 98.568	valid's rmse: 130.761
[2000]	train's rmse: 95.3729	valid's rmse: 129.568
[2200]	train's rmse: 92.4101	valid's rmse: 128.811
[2400]	train's rmse: 89.646	valid's rmse: 127.927
[2600]	train's rmse: 87.1639	valid's rmse: 127.01
[2800]	train's rmse: 84.8595	valid's rmse: 126.384
[3000]	train's rmse: 82.6762	valid's rmse: 125.863
[3200]	train's rmse: 80.551	valid's rmse: 125.651
[3400]	train's rmse: 78.6691	valid's rmse: 125.195
[3600]	train's rmse: 76.8198	valid's rmse: 124.758
[3800]	train's rmse: 75.0911

,y_true,forecast_solar_day_ahead,pred_mean_model2,pred_p16_model2,pred_p50_model2,pred_p84_model2,generation solar_roll3h,generation solar_roll6h,generation solar_lag1h,generation solar_lag2h,generation solar_lag3h,generation solar_lag24h
utc_time,,,,,,,,,,,,
2017-10-19 15:00:00,1787.0,1457.0,1762.818839,1478.678679,1804.963445,1827.104693,2354.333333,2361.000000,2156.000000,2356.000000,2551.000000,805.0
2017-10-19 16:00:00,998.0,678.0,956.047401,818.078103,936.729936,1177.613862,2105.654482,2338.660574,1804.963445,2156.000000,2356.000000,469.0
2017-10-19 17:00:00,256.0,202.0,252.337148,197.819168,228.338164,438.577105,1632.564460,2076.782230,936.729936,1804.963445,2156.000000,81.0
2017-10-19 18:00:00,59.0,108.0,109.167541,59.297974,113.274728,132.704007,990.010515,1672.171924,228.338164,936.729936,1804.963445,48.0
2017-10-19 19:00:00,44.0,68.0,67.657195,57.928289,71.571009,88.658790,426.114276,1265.884379,113.274728,228.338164,936.729936,41.0


In [28]:
# ----------------------------- Model 2 -- Solar generation using roll and lag features --------------------------

# ---------- Build plotting DataFrame with bands ----------
plot_roll_quant = pd.DataFrame({
    'Actual Solar': y_all.values,
    'LightGBM 24h Prediction Mean': y_pred_mean_full.values,
    'LightGBM 24h Prediction Median': y_pred_p50_full.values,
    'LightGBM Lower Band': y_pred_p16_full.values,
    'LightGBM Upper Band': y_pred_p84_full.values,
    'Forecast Solar Day Ahead': forecast_all.values
}, index=X_all.index)

plot_roll_quant.index.name = 'utc_time'

# Align source dataframe to the same hourly index
base_df_aligned = (
    data_lag_roll_solar
    .copy()
)
base_df_aligned[datetime_col] = pd.to_datetime(base_df_aligned[datetime_col])
base_df_aligned = (
    base_df_aligned
    .sort_values(datetime_col)
    .set_index(datetime_col)
    .reindex(X_all.index)
)

# 1) Add normal feature columns (e.g. GHI_weighted, etc.) with proper alignment
plot_roll_quant = plot_roll_quant.join(base_df_aligned[features_cols], how='left')

# 2) Build full lag/roll series for plotting:
#    - start with the actual/precomputed lag-roll values everywhere
#    - overwrite forecast-period rows with recursive values used by the model
target_features_plot = base_df_aligned[target_related_col].copy()

# overwrite only where recursive values exist (forecast windows)
target_features_plot.update(recursive_target_features_full)

# join the final train+forecast lag/roll features used for plotting
plot_roll_quant = plot_roll_quant.join(target_features_plot, how='left')

# Optional: also keep separate columns so you can compare actual-vs-recursive lag/roll directly
actual_target_features = base_df_aligned[target_related_col].copy()
actual_target_features = actual_target_features.add_suffix('_actual')

recursive_target_features_plot = recursive_target_features_full.copy()
recursive_target_features_plot = recursive_target_features_plot.add_suffix('_recursive')

plot_roll_quant = plot_roll_quant.join(actual_target_features, how='left')
plot_roll_quant = plot_roll_quant.join(recursive_target_features_plot, how='left')

# Optional: mark where forecasting begins
plot_roll_quant['is_forecast_period'] = 0
plot_roll_quant.loc[y_pred_mean_full.dropna().index, 'is_forecast_period'] = 1

# Back to a normal column for your plotting function
plot_roll_quant = plot_roll_quant.reset_index()

# Quick checks
print(plot_roll_quant[['generation solar_roll3h', 'generation solar_lag3h', 'generation solar_lag24h', 'GHI_weighted']].notna().sum())
display(plot_roll_quant[['utc_time', 'generation solar_roll3h', 'generation solar_lag3h', 'generation solar_lag24h',
                         'GHI_weighted']].head())
display(plot_roll_quant[['utc_time', 'generation solar_roll3h', 'generation solar_lag3h', 'generation solar_lag24h',
                         'GHI_weighted']].tail())

# ---------- 5) Main plot (your existing function; just pass CI columns) ----------
# Draw the single clean plot over whole test span (your function supports CI via test_pred_col/lower/upper)
p = ML_future_energy_predict(
    dataframe=plot_roll_quant,
    x_axis_column='utc_time', 
    y_axis_columns=['Actual Solar', 'LightGBM 24h Prediction Mean', 'LightGBM 24h Prediction Median', 'Forecast Solar Day Ahead'],
    x_axis_label=r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$",
    y_axis_label=r"$$\mathrm{Solar\ Power\ Generation\ (MWh)}$$",
    title='Actual vs LightGBM 24h Rolling Prediction (Mean and Median) with 1 sigma confidence bounds vs Day-Ahead Forecast',
    feature_columns=(['generation solar_roll3h', 'generation solar_lag3h', 'generation solar_lag24h', 'GHI_weighted']
                     if 'generation solar_roll3h' in plot_roll_quant.columns and 'generation solar_lag3h' in plot_roll_quant.columns and
                     'generation solar_lag24h' in plot_roll_quant.columns and 'GHI_weighted' in plot_roll_quant.columns else None),
    features_ylabel=[r"$$\mathrm{Solar\ Power\ Generation\ roll/lag\ (MWh)\ \&\ GHI\ weighted\ (W\ m^{-2})}$$"] \
                     if 'generation solar_roll3h' in plot_roll_quant.columns and 'generation solar_lag3h' in plot_roll_quant.columns and \
                     'generation solar_lag24h' in plot_roll_quant.columns and 'GHI_weighted' in plot_roll_quant.columns else None,
    p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'pink', 'blue'],
    color_features=['green', 'cyan', 'black', 'violet'],
    labels=['Actual Solar', 'LightGBM 24h Prediction (Mean)', 'LightGBM 24h Prediction Median', 'Forecast Solar Day Ahead'],
    features_labels=(['Solar Gen Roll 3h', 'Solar Gen lag 3h', 'Solar Gen lag 24h', 'Weighted GHI'] 
                     if 'generation solar_roll3h' in plot_roll_quant.columns and 'generation solar_lag3h' in plot_roll_quant.columns and
                     'generation solar_lag24h' in plot_roll_quant.columns and 'GHI_weighted' in plot_roll_quant.columns else None),
    symbols=['star', 'triangle', 'inverted_triangle', 'diamond'],
    symbols_features=(['circle', 'square', 'square_pin', 'plus']
                      if 'generation solar_roll3h' in plot_roll_quant.columns and 'generation solar_lag3h' in plot_roll_quant.columns and
                     'generation solar_lag24h' in plot_roll_quant.columns and 'GHI_weighted' in plot_roll_quant.columns else None),
    test_pred_col='LightGBM 24h Prediction Mean',
    lower_conf_col='LightGBM Lower Band',
    upper_conf_col='LightGBM Upper Band')

show(p)

# Save plots.
direct_out = current_dir + '/output/results/LightGBM/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_Solar_Power_CAMS_LightGBM_24hr_step_ahead_CV_windows_lag_roll.html'

title = 'Actual vs LightGBM 24h Rolling Prediction (Mean and Median) with 1 sigma confidence bounds vs Day-Ahead Forecast'

save(p, filename_out, title=title)

generation solar_roll3h    35064
generation solar_lag3h     35064
generation solar_lag24h    35064
GHI_weighted               35064
dtype: int64


,utc_time,generation solar_roll3h,generation solar_lag3h,generation solar_lag24h,GHI_weighted
0,2014-12-31 23:00:00,49.000000,49.0,49.0,0.0
1,2015-01-01 00:00:00,49.000000,49.0,49.0,0.0
2,2015-01-01 01:00:00,49.500000,49.0,49.0,0.0
3,2015-01-01 02:00:00,49.666667,49.0,49.0,0.0
4,2015-01-01 03:00:00,50.000000,50.0,49.0,0.0


,utc_time,generation solar_roll3h,generation solar_lag3h,generation solar_lag24h,GHI_weighted
35059,2018-12-31 18:00:00,1862.444293,3172.958347,31.0,0.575652
35060,2018-12-31 19:00:00,914.228902,1810.018403,31.0,0.000000
35061,2018-12-31 20:00:00,381.880127,604.356130,31.0,0.000000
35062,2018-12-31 21:00:00,224.429270,328.312174,31.0,0.000000
35063,2018-12-31 22:00:00,145.971318,212.972078,30.0,0.000000


/var/folders/wv/ww_f6bg15tv52kq7ghvh0w880000gp/T/ipykernel_95818/922161625.py:113: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(p, filename_out, title=title)


'/Users/u8010412/Library/CloudStorage/Dropbox/Data_Science/Projects/Kaggle_energy_data/archive/output/results/LightGBM/plots/Predicted_Solar_Power_CAMS_LightGBM_24hr_step_ahead_CV_windows_lag_roll.html'

In [30]:
# (Optional) RMSE Skill Score vs Forecast
def rmse_skill(y_true, y_model, y_bench):
    rmse_m = np.sqrt(mean_squared_error(y_true, y_model))
    rmse_b = np.sqrt(mean_squared_error(y_true, y_bench))
    return 1.0 - (rmse_m / rmse_b) if rmse_b > 0 else np.nan

In [31]:
# ----------------------------- Model 2 -- Solar generation using roll and lag features --------------------------
# ========= Overall metrics on the full TEST span (LightGBM 24h mean/median vs Forecast) =========
# Assumes you already have: plot_roll_quant, test_start_time, test_end_time,
# and your helpers: safe_metrics, rmse_skill.

# 1) Slice the test range
mask_test = (plot_roll_quant['utc_time'] >= test_start_time) & (plot_roll_quant['utc_time'] <= test_end_time)

yt = plot_roll_quant.loc[mask_test, 'Actual Solar'].values
yp_mean = plot_roll_quant.loc[mask_test, 'LightGBM 24h Prediction Mean'].values
yp_median = plot_roll_quant.loc[mask_test, 'LightGBM 24h Prediction Median'].values
yf = plot_roll_quant.loc[mask_test, 'Forecast Solar Day Ahead'].values

# 2) Build metrics dict (LightGBM Mean + Median + Forecast) incl. RMSE Skill vs Forecast
metrics_overall = {
    "All Test Data": {
        "LightGBM 24h Mean": safe_metrics(yt, yp_mean) | {
            "RMSE Skill (vs Forecast)": rmse_skill(yt, yp_mean, yf)
        },
        "LightGBM 24h Median": safe_metrics(yt, yp_median) | {
            "RMSE Skill (vs Forecast)": rmse_skill(yt, yp_median, yf)
        },
        "Forecast": safe_metrics(yt, yf) | {
            "RMSE Skill (vs Forecast)": rmse_skill(yt, yf, yf)
        },
    }
}

# 3) Pretty DataFrame (MultiIndex columns)
df_overall = pd.DataFrame({
    (section, model): metrics_overall[section][model]
    for section in metrics_overall
    for model in metrics_overall[section]
})

# Optional: column order
df_overall = df_overall[[('All Test Data','LightGBM 24h Mean'),
                         ('All Test Data','LightGBM 24h Median'),
                         ('All Test Data','Forecast')]]

print("\n=== Overall Test Metrics (LightGBM 24h Mean/Median vs Forecast) ===")
print(df_overall.round(3))

# 4) Save CSV / TXT / Markdown
direct_out = current_dir + '/output/results/LightGBM'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/lightgbm_rolling24h_vs_forecast_metrics_overall_with_median_CAMS_roll_lag.csv'

df_overall.to_csv(filename_out)

filename_out = direct_out + '/lightgbm_rolling24h_vs_forecast_metrics_overall_with_median_CAMS_roll_lag.txt'
with open(filename_out, 'w', encoding='utf-8') as f:
    f.write("LightGBM 24h Rolling Prediction (Mean & Median) vs Day-Ahead Forecast — Overall Test Metrics\n")
    f.write("=" * 94 + "\n\n")
    f.write(df_overall.round(3).to_string())
    f.write("\n")

filename_out = direct_out + '/lightgbm_rolling24h_vs_forecast_metrics_overall_with_median_CAMS_roll_lag.md'
with open(filename_out, 'w', encoding='utf-8') as f:
    f.write("# LightGBM 24h Rolling Prediction (Mean & Median) vs Day-Ahead Forecast — Overall Test Metrics\n\n")
    md_df = df_overall.copy()
    md_df.columns = [f"{lvl0} – {lvl1}" for (lvl0, lvl1) in md_df.columns]
    f.write(md_df.round(3).to_markdown(index=True))
    f.write("\n")

#print(f"\nSaved overall metrics (mean & median) to:\n- {csv_path}\n- {txt_path}\n- {md_path}")


=== Overall Test Metrics (LightGBM 24h Mean/Median vs Forecast) ===
                             All Test Data                               
                         LightGBM 24h Mean LightGBM 24h Median   Forecast
MSE                             306762.767          309803.592  46701.181
RMSE                               553.862             556.600    216.105
MAE                                281.214             282.836    130.143
MAPE                               143.337             145.226     54.601
sMAPE                               48.223              48.440     36.964
R2                                   0.886               0.885      0.983
RMSE Skill (vs Forecast)            -1.563              -1.576      0.000


In [26]:
# ----------------------------- Model 2 -- Solar generation using roll and lag features --------------------------

# Work on later as I need to run the LightGBM model, but using the traditional train/test split.

# --- 6) Feature importance
# Importance by gain and by split (counts)
feat_names = mean_model.feature_name()
imp_gain   = mean_model.feature_importance(importance_type='gain')
imp_split  = mean_model.feature_importance(importance_type='split')

fi_df = (pd.DataFrame({
            'feature': feat_names,
            'gain': imp_gain,
            'split': imp_split
        })
        .assign(gain_pct=lambda d: 100 * d['gain'] / d['gain'].sum() if d['gain'].sum() > 0 else 0)
        .sort_values('gain', ascending=False)
        .reset_index(drop=True))

print("\nTop 20 features by gain:\n", fi_df.head(20))

filename_out = direct_out + '/lgbm_feature_importance_gain_split_Solar_CAMS.csv'
fi_df.to_csv(filename_out, index=False)

                 utc_time  Actual Solar  LightGBM 24h Prediction Mean  \
0     2014-12-31 23:00:00          49.0                           NaN   
1     2015-01-01 00:00:00          50.0                           NaN   
2     2015-01-01 01:00:00          50.0                           NaN   
3     2015-01-01 02:00:00          50.0                           NaN   
4     2015-01-01 03:00:00          42.0                           NaN   
...                   ...           ...                           ...   
35059 2018-12-31 18:00:00          85.0                    285.884429   
35060 2018-12-31 19:00:00          33.0                    210.760242   
35061 2018-12-31 20:00:00          31.0                    107.400878   
35062 2018-12-31 21:00:00          31.0                     92.574048   
35063 2018-12-31 22:00:00          31.0                     73.383605   

       LightGBM 24h Prediction Median  LightGBM Lower Band  \
0                                 NaN                  NaN   

In [32]:
import dill
import inspect

def is_picklable(obj) -> bool:
    try:
        dill.dumps(obj)
        return True
    except Exception:
        return False

def is_bokeh_object(obj) -> bool:
    return obj.__class__.__module__.startswith("bokeh.")

skip_names = {"In", "Out", "exit", "quit", "get_ipython", "dill"}

bad_names = []
session_vars = {}

for name, val in list(globals().items()):
    if name in skip_names or name.startswith("_"):
        continue
    if inspect.ismodule(val) or inspect.isfunction(val):
        continue
    if is_bokeh_object(val):
        bad_names.append((name, "bokeh object"))
        continue
    if is_picklable(val):
        session_vars[name] = val
    else:
        bad_names.append((name, type(val).__name__))

print(bad_names)

[('Label', 'bokeh object'), ('ColumnDataSource', 'bokeh object'), ('CustomJS', 'bokeh object'), ('Slider', 'bokeh object'), ('Whisker', 'bokeh object'), ('BoxAnnotation', 'bokeh object'), ('Arrow', 'bokeh object'), ('OpenHead', 'bokeh object'), ('Span', 'bokeh object'), ('figure', 'bokeh object'), ('Legend', 'bokeh object'), ('LinearAxis', 'bokeh object'), ('Range1d', 'bokeh object'), ('LabelSet', 'bokeh object'), ('HoverTool', 'bokeh object'), ('DatetimeTickFormatter', 'bokeh object'), ('dtr', 'Dataset'), ('dva', 'Dataset'), ('p', 'bokeh object')]


In [33]:
# Now try dumping the session
with open("notebook_session.pkl", "wb") as f:
    dill.dump(session_vars, f)

In [ ]:
import dill

# Restore the notebook session
with open("notebook_session.pkl", "rb") as f:
    session_vars = dill.load(f)

globals().update(session_vars)